# No Libraries, No Shortcuts: LLM from Scratch with PyTorch

## The no-BS guide to build, train, and fine-tune a Transformer architecture from scratch

<p align="center">
  <img src="images/intro.png" alt="Intro image" width="200"/>
</p>


OpenAI has recently launched its highly anticipated open-source GPT-OSS models, a moment that invites a minute of reflection on just how far we’ve come. Years back, even before ChatGPT, I remember reading an article on a GPT model, probably GPT-2, that writes its own essays and poems, and they were just experiments. Fast forward to today, it has already become an integral part of my daily life. And it all started with the landmark "Attention is All You Need" publication in 2017 by Google Research. The Transformer architecture was proposed, which soon powered the very first GPT — GPT-1 (Generative Pretrained Transformer) in 2018.

In the eight years since, progress has been nothing short of extraordinary. Modern large language models now boast multimodal capabilities, advanced reasoning skills, and innovative architectural refinements. Yet, at their core, they still rely on the Transformer skeleton. For many developers today, the intricate brilliance of this underlying design often goes unnoticed, thanks to the accessibility of LLMs today through user-friendly frameworks and APIs.

In this article, we will break apart the Transformer architecture and understand it piece by piece. By the end, you’ll have built your own LLM from scratch, which writes brand-new Coldplay songs (who knows, you might just start composing lyrics for Coldplay!).


This notebook is a cleaned, course-style rewrite of the article’s walkthrough. It follows the same logical flow:

>Introduction 
→ Transformer intuition 
→ tokenization 
→ embeddings 
→ positional encoding 
→ multi-head attention 
→ feed-forward network
→ encoder/decoder blocks 
→ full Transformer assembly 
→ training loop 
→ loss & optimization.


## Table of Contents

- [A Quick Recap](#a-quick-recap)
  - [Tokenizer](#tokenizer)
  - [Next Token Predictors](#next-token-predictors)
- [Attention is All You Need](#attention-is-all-you-need)
- [Building the Transformer Architecture](#building-the-transformer-architecture)
  - [Tokenization](#tokenization)
  - [Positional Encoding & Embeddings](#positional-encoding--embeddings)
  - [Self-Attention: How Tokens Gossip About Each Other](#self-attention-how-tokens-gossip-about-each-other)
  - [Multi-Head Attention: The Group Chat in Your Model’s Brain](#multi-head-attention-the-group-chat-in-your-models-brain)
  - [Feed-Forward Networks](#feed-forward-networks)
  - [The Decoder with Residual Connections](#the-decoder-with-residual-connections)
  - [Putting It All Together: The Transformer Skeleton](#putting-it-all-together-the-transformer-skeleton)
- [Model Pretraining](#model-pretraining)
  - [Data Preparation](#data-preparation)
  - [Training](#training)
- [Teaching Your Model to Follow (and Sing Coldplay) — Fine-Tuning](#teaching-your-model-to-follow-and-sing-coldplay--fine-tuning)
- [Wrapping Up](#wrapping-up)
- [References](#references)


## Introduction to LLMs

A large language model (LLM) looks like it “understands” text, but the core training objective is simple:
**predict the next token**.

If we represent a piece of text as a sequence of tokens:

$$t_1, t_2, \dots, t_n$$

then an autoregressive (causal) language model learns the conditional distribution:

$$p(t_i \mid t_1, \dots, t_{i-1})$$

During training, we show the model many examples and push it to assign high probability to the correct next token.
During generation, we repeatedly apply the model and append predicted tokens one-by-one.

### Tokenizer

Neural networks can’t directly process raw text. We first convert text into **token IDs** (integers).
That conversion is handled by the tokenizer. Depending on design, tokens can be:

- words
- subwords
- characters

Example: “Hold my math!”

- Word-level: `["Hold", "my", "math", "!"]`
- Subword-level: `["Hold", "my", "ma", "th", "!"]`
- Character-level: `["H","o","l","d"," ","m","y"," ","m","a","t","h","!"]`

In this notebook we implement a *character-level* tokenizer (fully from scratch) because it’s the most transparent for learning.

### Next-Token Predictors

Given a prefix of tokens, a language model outputs a probability distribution over the vocabulary for the next token.

![Next-token predictor](images/next_token_predictor.png)

Generation is simply repeating the same step in a loop: predict → sample → append.

![Autoregressive window](images/autoregressive_window.png)

Practical models can only process a fixed maximum number of tokens at once (the **context window**),
so generation always conditions on a bounded window of previous tokens.


### A quick note on chat formatting

Chat APIs often accept a list of messages like `{role, content}`. Internally, many systems serialize these into
a single text prompt with special boundary markers. The exact markers vary by model family, but the idea is the same:
teach the model to separate **system** behavior, **user** queries, and **assistant** responses.


In [1]:
messages = [
    {"role": "system", "content": "You are a creative storyteller."},
    {"role": "user", "content": "Write a creative story"},
]

serialized = (
    "<|im_start|>system\n"
    "You are a creative storyteller.\n"
    "<|im_end|>\n"
    "<|im_start|>user\n"
    "Write a creative story\n"
    "<|im_end|>\n"
    "<|im_start|>assistant\n"
)

messages, serialized


([{'role': 'system', 'content': 'You are a creative storyteller.'},
  {'role': 'user', 'content': 'Write a creative story'}],
 '<|im_start|>system\nYou are a creative storyteller.\n<|im_end|>\n<|im_start|>user\nWrite a creative story\n<|im_end|>\n<|im_start|>assistant\n')

## Attention is All You Need

So how do large language models decide what to generate?

At first glance, a model has the freedom to generate almost anything, which could lead to random or meaningless outputs. To produce coherent and relevant text, neural networks must learn to use context not only from the last token but from the entire input sequence.

Consider the example:

**Input:**  
The cat chased the

If the model only looks at the last token ("the"), it might predict almost anything such as "banana", "man", or "moon".

However, if it considers the full context — "The", "cat", "chased", "the" — it can infer that **"mouse"** is far more likely than unrelated words.

---

### Context Matters Beyond Completion

This becomes even more important in tasks like translation, where relationships between words are not always one-to-one.

**English:**  
I eat a red apple.

**French:**  
Je mange une pomme rouge.

**Word alignment:**

English:  [I]     [eat]     [a]      [red]     [apple]  
      ↓        ↓        ↓         ↓          ↓  
French:   [Je]   [mange]   [une]    [rouge]    [pomme]  

| English | I | eat | a | red | apple |
|---------|---|-----|---|-----|-------|
| French  | Je | mange | une | rouge | pomme |

Notice that the word **"red" (rouge)** appears *after* "apple" (pomme) in French.

---

### Why This Matters

A simple word-by-word translation does not work here. Instead, the model must learn:

- Relationships between words  
- Context across the entire sentence  
- Flexible word ordering across languages  

Traditional sequence models struggled with this.

The Transformer solves it using **attention**, where every token can directly interact with every other token. This allows the model to capture global context efficiently and generate meaningful outputs.

## Transformer Intuition

Early sequence models like RNNs process tokens one at a time, passing a hidden state forward.
That helps incorporate “past context”, but creates two big problems:

1. **Long-range dependencies are fragile**: older information can fade as sequences get longer.
2. **Training is hard to parallelize**: each step depends on the previous step.

![RNN overview](images/rnn.png)

LSTMs (with gates) improved memory retention, but the sequential bottleneck still remains.

![LSTM overview](images/lstm.png)

Transformers replace recurrence with **attention**, which connects every token to every other token directly.
Instead of “carrying memory forward”, each token computes how much it should “pay attention” to other tokens.

### The attention computation

Given token representations $X \in \mathbb{R}^{T \times d_{model}}$, we project each token into:

- Query $Q$ (what I’m looking for)
- Key $K$ (what I offer)
- Value $V$ (what I will contribute if attended to)

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

Then scaled dot-product attention is:

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

**💡 Insight**

Attention is content-based routing: tokens choose what information to mix in by comparing queries to keys,
and then averaging values using those similarity scores.


In [2]:
import math
import random
from dataclasses import dataclass
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int = 1337) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(1337)
print("device:", device)


device: cpu


## Tokenization

We’ll implement a character-level tokenizer:

- Build a vocabulary from the training corpus.
- Encode: text → list of integers.
- Decode: list of integers → text.

This keeps the mechanics crystal clear and requires no external tokenization libraries.


In [3]:
class CharTokenizer:
    def __init__(self, text: str, add_special_tokens: bool = True):
        self.special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"] if add_special_tokens else []

        chars = sorted(list(set(text)))
        self.itos = self.special_tokens + chars
        self.stoi = {ch: i for i, ch in enumerate(self.itos)}

        self.pad_id = self.stoi.get("<pad>")
        self.unk_id = self.stoi.get("<unk>")
        self.bos_id = self.stoi.get("<bos>")
        self.eos_id = self.stoi.get("<eos>")

    @property
    def vocab_size(self) -> int:
        return len(self.itos)

    def encode(self, s: str, add_bos: bool = False, add_eos: bool = False) -> List[int]:
        ids: List[int] = []
        if add_bos and self.bos_id is not None:
            ids.append(self.bos_id)
        for ch in s:
            ids.append(self.stoi.get(ch, self.unk_id))
        if add_eos and self.eos_id is not None:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids: List[int], strip_special: bool = True) -> str:
        out: List[str] = []
        for i in ids:
            if i < 0 or i >= len(self.itos):
                continue
            tok = self.itos[i]
            if strip_special and tok in self.special_tokens:
                continue
            out.append(tok)
        return "".join(out)

sample = "Hold my math!"
tok = CharTokenizer(sample)
encoded = tok.encode(sample, add_bos=True, add_eos=True)
print("vocab_size:", tok.vocab_size)
print("encoded:", encoded)
print("decoded:", tok.decode(encoded))


vocab_size: 15
encoded: [2, 6, 12, 10, 8, 4, 11, 14, 4, 11, 7, 13, 9, 5, 3]
decoded: Hold my math!


## Embeddings

Token IDs are discrete integers; Transformers operate on vectors.
So we learn a token embedding table $E \in \mathbb{R}^{V \times d_{model}}$, where:

- $V$ is vocabulary size
- $d_{model}$ is the embedding dimension

Each token id indexes a row in $E$ to produce a vector representation.


In [4]:
V = tok.vocab_size
D = 16
embedding = nn.Embedding(V, D)

ids = torch.tensor(tok.encode(sample), dtype=torch.long)
vecs = embedding(ids)

print("ids shape:", tuple(ids.shape))
print("vecs shape:", tuple(vecs.shape))


ids shape: (13,)
vecs shape: (13, 16)


## Positional Encoding

Without recurrence or convolution, a Transformer has no built-in notion of order.
We inject position information via **positional encodings**.

The sinusoidal encoding from the original Transformer paper is:

$$PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

We add positional encodings to token embeddings:

$$X = \mathrm{Embed}(tokens) + PE$$

so the model can reason about both meaning and order.


In [5]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = x.size(1)
        return x + self.pe[:, :T, :]

pos_enc = SinusoidalPositionalEncoding(16, max_len=64)
print(pos_enc(torch.zeros(2, 10, 16)).shape)


torch.Size([2, 10, 16])


## Multi-Head Attention

Self-attention works by comparing every token to every other token.
For a single head:

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

**Why the scaling term?**
The dot product $QK^\top$ grows with $d_k$. Scaling by $\sqrt{d_k}$ keeps softmax in a healthy range
and prevents tiny gradients.

**Why multiple heads?**
Multi-head attention runs several attention computations in parallel, each on a smaller subspace.
Different heads can specialize in different patterns (syntax, long-range dependencies, matching brackets, etc.).

**⚠️ Common Mistakes**

- Forgetting the causal mask for language modeling (it leaks future tokens).
- Applying softmax over the wrong axis (must normalize over key positions).
- Mixing up the `(B, T, h, d_head)` reshapes.


In [6]:
def make_causal_mask(T: int, device=None) -> torch.Tensor:
    m = torch.tril(torch.ones(T, T, dtype=torch.bool, device=device))
    return m.view(1, 1, T, T)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0, is_causal: bool = False):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.is_causal = is_causal

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        kv: torch.Tensor | None = None,
        attn_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        B, Tq, _ = x.shape
        if kv is None:
            kv = x
        _, Tk, _ = kv.shape

        q = self.q_proj(x)
        k = self.k_proj(kv)
        v = self.v_proj(kv)

        q = q.view(B, Tq, self.n_heads, self.d_head).transpose(1, 2)  # (B, h, Tq, d_head)
        k = k.view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)  # (B, h, Tk, d_head)
        v = v.view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)  # (B, h, Tk, d_head)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)  # (B, h, Tq, Tk)

        if self.is_causal:
            causal = make_causal_mask(Tq, device=scores.device)
            scores = scores.masked_fill(~causal, float("-inf"))

        if attn_mask is not None:
            scores = scores.masked_fill(~attn_mask, float("-inf"))

        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = attn @ v  # (B, h, Tq, d_head)
        out = out.transpose(1, 2).contiguous().view(B, Tq, self.d_model)
        return self.out_proj(out)

mha = MultiHeadAttention(d_model=32, n_heads=4, dropout=0.1, is_causal=True)
print(mha(torch.randn(2, 8, 32)).shape)


torch.Size([2, 8, 32])


**Explanation**

- We project inputs into Q/K/V of shape `(B, T, d_model)`.
- We split `d_model` into `n_heads` smaller heads of size `d_head`.
- We compute attention scores for each head: `(B, h, Tq, Tk)`.
- Softmax runs over `Tk` so weights sum to 1 across keys for each query token.
- A causal mask ensures token *i* can’t attend to tokens *j > i*.


## Feed Forward Network

Attention mixes information across tokens. The feed-forward network (FFN) then transforms each token’s representation independently.

$$\mathrm{FFN}(x) = W_2\,\sigma(W_1x + b_1) + b_2$$

Typically the hidden dimension is ~4× larger than $d_{model}$. We’ll use GELU activation.


In [7]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

print(FeedForward(32, 128, 0.1)(torch.randn(2, 8, 32)).shape)


torch.Size([2, 8, 32])


## Encoder & Decoder Blocks

A full “classic” Transformer has:

- **Encoder blocks** (self-attention + FFN)
- **Decoder blocks** (masked self-attention + cross-attention + FFN)

Many LLMs (GPT-style) are **decoder-only** (masked self-attention + FFN) because next-token prediction needs only causal context.

We’ll implement both:

- an encoder block (for completeness and conceptual alignment)
- a decoder block (used by our language model)

We’ll use **Pre-LN** (LayerNorm before each sub-layer), which is common in modern training for stability.


In [8]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout=dropout, is_causal=False)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout=dropout, is_causal=True)

        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout=dropout, is_causal=False)

        self.ln3 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout=dropout)

    def forward(self, x: torch.Tensor, enc_out: torch.Tensor | None = None) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        if enc_out is not None:
            x = x + self.cross_attn(self.ln2(x), kv=enc_out)
        x = x + self.ffn(self.ln3(x))
        return x


## Full Transformer Assembly

For language modeling, we build a **decoder-only Transformer**:

1. Token embedding + positional encoding
2. Stack of masked decoder blocks
3. Output projection to vocabulary size

The model produces logits:

$$\text{logits} \in \mathbb{R}^{B \times T \times V}$$

and training uses cross-entropy against the shifted targets.


In [9]:
@dataclass
class TransformerLMConfig:
    vocab_size: int
    d_model: int = 192
    n_heads: int = 6
    n_layers: int = 4
    d_ff: int = 768
    dropout: float = 0.1
    max_len: int = 128

class TransformerDecoderOnlyLM(nn.Module):
    def __init__(self, cfg: TransformerLMConfig):
        super().__init__()
        self.cfg = cfg
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_enc = SinusoidalPositionalEncoding(cfg.d_model, max_len=cfg.max_len)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList(
            [DecoderBlock(cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout) for _ in range(cfg.n_layers)]
        )
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        self.lm_head.weight = self.token_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        assert T <= self.cfg.max_len
        x = self.token_emb(idx)
        x = self.pos_enc(x)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x, enc_out=None)
        x = self.ln_f(x)
        return self.lm_head(x)

cfg = TransformerLMConfig(vocab_size=tok.vocab_size, max_len=128)
model = TransformerDecoderOnlyLM(cfg).to(device)
test_ids = torch.tensor([tok.encode("Hold my m")[:20]], dtype=torch.long, device=device)
print("logits:", model(test_ids).shape)


logits: torch.Size([1, 9, 15])


## Training Loop

### Data preparation

For causal LM training, we create examples from a long token stream using a fixed block size `T`:

- input `x`: `[t_1, ..., t_T]`
- target `y`: `[t_2, ..., t_{T+1}]`

So the model learns to predict the next token at every position in parallel.


In [10]:
from torch.utils.data import D=
ataset, DataLoaderclass CustomDataset(Dataset):    def __init__(self, txt, tokenizer, max_length, stride):        self.=
input_ids =3D []        self.target_ids =3D []        # Tokenize the entire text        token_ids =
=3D tokenizer.encode(txt, add_special_tokens=3DFalse)        # Use a sliding =
window to chunk the data into overlapping sequences of max_length        for i in range(0, len(token_ids) - =
max_length, stride):            input_chunk =3D token_ids[i:i + max_len=
gth]            target_chunk =3D token_ids[i + 1: i + max_length + 1]   =
         self.input_ids.append(torch.tensor(input_chunk))            se=
lf.target_ids.append(torch.tensor(target_chunk))    def __len__=
(self):        return len(self.input_i=
ds)    def __getitem__(self, idx):        return self.input_ids=
[idx], self.target_ids[idx]def create_encoded_dataloader(txt, tokenizer, batch_size=3D4, max_length=3D128, =
                        stride=3D128, sh=
uffle=3DTrue, drop_last=3DTrue, num_workers=3D0):    # Create dataset    dataset =3D CustomDataset(txt, tokenizer, max_length, stride)    # Create dataloader    da=
taloader =3D DataLoader(        dataset, batch_size=3Dbatch_size, shuff=
le=3Dshuffle, drop_last=3Ddrop_last, num_workers=3Dnum_workers, pin_memory=
=3DTrue)    return dataloadertotal_characters =3D len(train_text_data)total_tokens =3D len(tokenizer.encode(train_text_data))=
print("Cha=
racters:", total_characters)print("Tokens:", total_tokens)# Sanity checkif total_tokens * (0.95) &lt; context_length:    print("Not enough tokens for the training loader. "          "Try to lower the context_len=
gth or "          "increase the `tra=
ining_ratio`")if total_t=
okens * (1-0=
.95) &lt;context_length:    print("Not enough tokens for the validation lo=
ader. "          "Try to lower the c=
ontext_length or "          "decreas=
e the `training_ratio`")train_dataloader =3D create_enco=
ded_dataloader(    train_text_data,    tokenizer=3Dtokenizer,  =
  batch_size=3D2,    max_length=3Dco=
ntext_length,    stride=3Dcontext_length,    shuffle=3DTrue,    drop_last=3DTrue)test_dataloader =3D create_encoded_dataloader(    test_text_data,    tokenizer=3Dtokenizer,    batch_size=3D2,    max_length=3Dcontext_length,=
    stride=3Dcontext_length,    shuffle=3D=
False,    drop_last=3DTrue)


SyntaxError: invalid decimal literal (3080803006.py, line 3)

In [ ]:
input_ids: [101, 102, 103, 104, 105]untokenized input_ids: ["The", "cat", "sat", "on", =
"the"]target_ids: [102, 103, 10=
4, 105, 106]untokenized target_ids: ["cat", "sat", "on", "the", "mat"]


In [ ]:
from datasets import load_datasetimport re# Load datasetds =3D l=
oad_dataset("stanfordnlp/imdb")# Keep only English (ASCII) charactersdef keep_english_only(text):  =
  return re.sub(r"[^\x00-\x7F]+", "", text)# Clean and combine a list of textsdef combine_and_clean(text_list):    # Keep only English    =
cleaned_list =3D [keep_english_only(t) for t in text_list]    # Combine into one string    combined =3D " ".join(cleaned_list)    # Remove extra spaces/newlines    combined =3D re=
.sub(r'\s+', ' ', combined).strip()    return combined# Create separate combin=
ed stringstrain_text_data =3D combine_and_clean(ds['train']['text']=
)test_text_data =3D combine_and_clean(ds['t=
est']['text'])


In [ ]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, token_ids: List[int], block_size: int):
        super().__init__()
        self.data = torch.tensor(token_ids, dtype=torch.long)
        self.block_size = block_size

    def __len__(self) -> int:
        return max(0, len(self.data) - self.block_size - 1)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + 1 + self.block_size]
        return x, y

corpus = (
    "We were born to make history.\n"
    "Lights will guide you home, and ignite your bones.\n"
    "Nobody said it was easy.\n"
    "I used to rule the world.\n"
    "Look at the stars, look how they shine for you.\n"
) * 100

tokenizer = CharTokenizer(corpus)
all_ids = tokenizer.encode(corpus)

block_size = 128
split = int(0.9 * len(all_ids))
train_ids, val_ids = all_ids[:split], all_ids[split:]

train_ds = TextDataset(train_ids, block_size)
val_ds = TextDataset(val_ids, block_size)

batch_size = 64
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=True)

cfg = TransformerLMConfig(vocab_size=tokenizer.vocab_size, max_len=block_size)
model = TransformerDecoderOnlyLM(cfg).to(device)

sum(p.numel() for p in model.parameters())


## Loss Function & Optimization

Next-token prediction uses **cross-entropy loss**.
If logits for a token are $z \in \mathbb{R}^{V}$ and the true next token is $y$:

$$\mathcal{L} = -\log(\mathrm{softmax}(z)_y)$$

In practice we compute cross-entropy over the whole batch and all time steps.
For optimization, **AdamW** is a widely used baseline for Transformers.


In [ ]:
def evaluate_loss(model: nn.Module, loader, max_batches: int = 50) -> float:
    model.eval()
    losses = []
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= max_batches:
                break
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
            losses.append(loss.item())
    model.train()
    return float(sum(losses) / max(1, len(losses)))

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

def train(model: nn.Module, steps: int = 200, log_every: int = 50) -> None:
    model.train()
    it = iter(train_loader)

    for step in range(1, steps + 1):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(train_loader)
            x, y = next(it)

        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if step == 1 or step % log_every == 0:
            val_loss = evaluate_loss(model, val_loader, max_batches=30)
            print(f"step {step:4d} | train_loss {loss.item():.4f} | val_loss {val_loss:.4f}")

train(model)


**💡 Insight**

With a causal mask, *every* position becomes a supervised learning target. A single batch teaches many “next token” problems in parallel.

**⚠️ Common Mistakes**

- Not shifting targets (`y`) by one position.
- Forgetting `model.eval()` during validation (dropout changes the loss).
- Training without a causal mask (future-token leakage).


In [ ]:
@torch.no_grad()
def generate(
    model: nn.Module,
    tokenizer: CharTokenizer,
    prompt: str,
    max_new_tokens: int = 200,
    temperature: float = 1.0,
) -> str:
    model.eval()
    idx = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.cfg.max_len :]
        logits = model(idx_cond)[:, -1, :] / max(1e-8, temperature)
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)

    return tokenizer.decode(idx[0].tolist())

print(generate(model, tokenizer, prompt="Look at the ", max_new_tokens=200, temperature=0.9))


## Fine-Tuning (High Level)

Pretraining learns general language patterns from large corpora. Fine-tuning adapts a pretrained model to a specific goal.

### Instruction tuning

Fine-tuning on instruction/response data teaches the model to follow user requests instead of only continuing text.
Data is often formatted with role markers (system/user/assistant) so the model learns what kind of continuation is expected.

### Parameter-efficient fine-tuning

Instead of updating all parameters, methods like LoRA introduce small trainable adapters (low-rank matrices),
reducing memory requirements and helping preserve general pretrained knowledge.


## Towards AI
=C2=
=B7Follow publicationWe build Enterprise AI. We teach what we learn. Join 100K+ AI practitione=
rs on Towards AI Academy. Free: 6-day Agentic AI Engineering Email Guide: https://email-c=
ourse.towardsai.net/Follow publ=
ication=
Top highlight=
This member-only story is on us. Upgrade to access all of Medium.Member-only story



# No Libr=
aries, No Shortcuts: LLM from Scratch with PyTorch
The no B=
S guide to build, train, and fine-tune a Transformer architecture from scra=
tch=
Ashi=
sh AbrahamFollow23 min read=C2=B7Oct=
 2, 20251.3K17=
Listen=
ShareMore=
Press enter or clic=
k to view image in full sizeImage By AIOpenAI has recently launched its highly anticipated open-sour=
ce GPT-OSS models, a moment that invites a minute of reflection on just how=
 far we=E2=80=99ve come. Years back, even before ChatGPT, I remember readin=
g an article on a GPT model, probably GPT-2, that writes its own essays and=
 poems, and they were just experiments. Fast forward to today, it has alrea=
dy become an integral part of my daily life. And it all started with the la=
ndmark "Attention is All You Need" pub=
lication in 2017 by Google Research. The Transformer architecture was propo=
sed, which soon powered the very first GPT =E2=80=94 GPT-1 (Generative Pret=
rained Transformer) in 2018.In the eight years since, progr=
ess has been nothing short of extraordinary. Modern large language models n=
ow boast multimodal capabilities, advanced reasoning skills, and innovative=
 architectural refinements. Yet, at their core, they still rely on the Tran=
sformer skeleton. For many developers today, the intricate brilliance of th=
is underlying design often goes unnoticed, thanks to the accessibility of L=
LMs today through user-friendly frameworks and APIs.In this=
 article, we will break apart the Transformer architecture and understand i=
t piece by piece. By the end, you=E2=80=99ll have built your own LLM from s=
cratch, which writes brand-new Coldplay songs (who knows, you might just st=
art composing lyrics for Coldplay!).



## A Quick Recap
For new readers, le=
t's take a quick review of what we discussed in the previous blog.=



## Everything You Need To Know on LLMs : Brick by Brick


### A comprehensive study on=
 LLMs , explored layer-by-layer
medium.comTokenizerAny text you provide is broken down by the LLM=E2=80=99s token=
izer into smaller units called tokens, which can range from a single charac=
ter to an entire word.Consider the text: =
=E2=80=9CHold my math!=E2=80=9D Depending on the model=E2=80=99s desig=
n, tokens can be words, subwords, or even characters.Word-level tokenization:["Hold", "my", "math", "!"]Subword-level tokenization:["Hold", "my", "ma", "th", "!"]Character-level tokenization:["H", "o", "l", "d", " ", "m", "y", " ", "m", "a", "t", "h", "=
!"]



### Next Token Predictors
Large language models are, by basic=
 definition, next-token predictors. Given the input t=
okens, the model learns to analyze and predict the probability of what the =
next token can be.Image By Autho=
rThey handle only a fixed number of token=
s at once, generating one token per step. Long replies you see, come from r=
epeating this process efficiently. This is achieved by sliding the input wi=
ndow forward after each prediction repeatedly and stopping at an eos token or a certain length limit.Image By AuthorIn fact, you might be thinking that for LLM API calls, you wri=
te the code in this way.messages =3D [    {=
        "role": "system",        "content": "You are a creative storyteller."=
    },    {        "role": "user",        "content": "Write a creative story=
"    },]But after the library proce=
sses this, the input that goes to the large language model will be differen=
t."""&lt;|im_start|&gt;systemYou are a creat=
ive storyteller.&lt;|im_end|&gt;&lt;|im_start|&gt;userWrite a c=
reative story&lt;|im_end|&gt;&lt;|im_start|&gt;assistant"""After pretraining (you=E2=80=99ll see that in t=
he coming sections), the large language model is fine-tuned on instructions=
 to make it usable for human interactions. This is called instruction tunin=
g. Otherwise, the LLM will remain as a simple text generator. The above for=
mat is how input is passed to instruction-tuned models, although the tags l=
ike &lt;|im_start|&gt;user may diff=
er for some of them.



In [ ]:
"""&lt;|im_start|&gt;systemYou are a creat=
ive storyteller.&lt;|im_end|&gt;&lt;|im_start|&gt;userWrite a c=
reative story&lt;|im_end|&gt;&lt;|im_start|&gt;assistant"""


In [ ]:
messages =3D [    {=
        "role": "system",        "content": "You are a creative storyteller."=
    },    {        "role": "user",        "content": "Write a creative story=
"    },]


## Attention is All You Need
So how do LLMs decide=
 what to generate? It has the freedom to generate anything random. But how =
to make it generate output that makes sense? For that, neural networks need=
 to learn and use context from not only the last token but also from other =
parts of the input sentence.Consider an example input, =
 The cat chased the=
If the model only looked at the last token=
 ("the"), it might predict almost a=
nything: "banana", "man", "moon", etc.But if it uses the full con=
text ("The", "cat", "chased", "th=
e"), it knows "mouse" is far=
 more likely than "moon" or "banana".This needs to wor=
k not only in completion tasks but also in translations.Eng=
lish:I eat a red apple.French:Je mange une pomme rouge.The matching words are;English:  [I]     [eat]     [a]      [red]     [apple]=
           =E2=86=93        =E2=86=93        =E2=86=93         =
=E2=86=93          =E2=86=93French:   [Je]=
   [mange]   [une]    [rouge]    [pomme]    As you can guess, the word for red=
 should come after the apple in Fre=
nch. So a word-for-word translation does not work here. Instead, the model =
should learn and figure out that in such use cases, word order is changed l=
ike this.In the early days of sequence modeling, Recurrent Neural Networks (RNNs) were the go-to choice =
for handling text. They processed words one at a time in sequence, passing =
a =E2=80=9Chidden state=E2=80=9D forward so each step carried some memory o=
f the past. This worked for short dependencies, but important information f=
rom earlier words often faded as the sequence grew longer.Press enter or click to view image in full sizeRNN (Source)Long Short-Term Memory (LSTM) networks, introduced i=
n 1997, improved on this by using special gating mechanisms, namely the inp=
ut gate, forget gate, and output gate to control the flow of information. T=
hese gates decide which parts of the current input and past memory to keep,=
 update, or discard, enabling the network to maintain relevant information =
over much longer sequences.Press enter =
or click to view image in full size=
LSTM (Source)The attention mechani=
sm, introduced in 2017, in Attention Is Al=
l You Need, addressed this limitation. Instead of relying on sequential=
 processing, attention connects every word in the input directly to every o=
ther word, computing weights that determine how much each token should infl=
uence the prediction. This allows the model to capture long-range dependenc=
ies efficiently and in parallel. Let=E2=80=99s do a quick dive into how the=
 attention mechanism works.The goal is to measure how much =
each token in a sentence relates to, or influences, every other token. The =
resulting measure for a pair of tokens is called the attention score. Collecting these scores for all token pairs pro=
duces the attention matrix. For this, we n=
eed three components for each token.=
Query vectorIt represents what this token is looking for in ot=
her tokens. It is calculated by the cross product of the input embedding ve=
ctor and a trainable query matrix Wq.Key vectorIt represents what this t=
oken offers as searchable information. It is calculated by the cross produc=
t of the input embedding vector and a trainable key matrix Wk.Value vectorIt contains the actual information content that will be passed along if t=
he token is attended to. It is calculated by the cross product of the input=
 embedding vector and a trainable value matrix Wv.Press enter or cli=
ck to view image in full sizeImage By Author ( dimensions ar=
e for illustration only)The score of a to=
ken is calculated by taking the dot product of the query vector with the ke=
y vector of the respective word we=E2=80=99re scoring. For simplicity, let =
us consider each word in this sentence as a token: =E2=80=9CThe cat slept on the mat and it purred.=E2=80=9D So if we=E2=80=99re=
 processing the self-attention for the word =E2=80=9Cslept=E2=80=9D at posi=
tion 3, the first score would be the dot product of q3 and k1. The second s=
core would be the dot product of q3 and k2, and so on. This will give us a =
score for each token. Each score is normalized using a softmax activation and divided by the =
square root of the embedding dimension(d**0.5) to get the attention weight. Finally, multiply each value vecto=
r by its corresponding attention weight to get the context vector.Press enter or click to view image in full sizeImage By Author ( dimensions are for illustration only)These context vectors are combined as a matrix =
that we refer to as the attention matrix. In application, all these operati=
ons are computed as matrix operations. As you can see, there are higher val=
ues for the cells =E2=80=9Cit-cat=E2=80=9D and =E2=80=
=9Cpurred-cat=E2=80=9D, showing higher correlation am=
ong these.Press enter or click to view =
image in full sizeImage By AuthorIn theory, the probability of a token being chosen as the next one s=
hould depend on the past tokens only and not on the future tokens, or else =
it doesn't make sense. To implement this, we need to mask all the future va=
lues in the matrix during the training phase and adjust the row values so t=
hat they add up to one as earlier, because these values are probabilities. =
This is called Causal Attention, and you w=
ill understand it much better in the upcoming coding part.Press enter or click to view image in full sizeImage By Author



In [ ]:
English:  [I]     [eat]     [a]      [red]     [apple]=
           =E2=86=93        =E2=86=93        =E2=86=93         =
=E2=86=93          =E2=86=93French:   [Je]=
   [mange]   [une]    [rouge]    [pomme]


## Building the Transformer Architecture
Press enter or click to view image in full s=
izeTransformer Architecture Encoder + Decoder (Source)Before diving into code, let=E2=80=99s first unde=
rstand the big picture. A Transformer is built from encoder and decoder blocks, each made up of the same key componen=
ts: embeddings, positional encodings, self-attentio=
n, multi-head attention, and feed-forward layers.A=
t its core, think of it as a modular pipeline:Input sequence text is broken into tokens.Tokens are co=
nverted into numerical embeddings and tagged with positional encodings.Sel=
f-attention lets every token =E2=80=9Cpay attention=E2=80=9D to every other=
 token or in simpler words the model figures out how each token relates to =
others.Multi-head attention combines several self-attention layers and let=
s the model view those relationships from multiple angles.A feed-forward n=
etwork transforms the combined information into stronger features.By stacking many such layers, the model learns increasingly ri=
ch representations of language.Each of the sections below w=
ill peel apart one of these building blocks, and we=E2=80=99ll implement th=
em step by step in PyTorch until we have a working Transformer from scratch=
. You will focus on the GPT architecture, which is decoder-only.=
Sou=
rce



### Positional Encoding &amp; Embe=
ddings
Positional encoding encodes the location of each tok=
en within a sequence. Transformers rely on it to preserve ordering, since u=
nlike RNNs, they process all tokens in parallel. A simple index isn=E2=80=
=99t expressive enough for long sequences, so positional encoding uses math=
ematical patterns (often sine and cosine functions) to create a matrix that=
 embeds richer positional information. This gives the model a sense of sequ=
ence order while still leveraging parallel computation.=
Image By AuthorThe embedding layer is t=
he first step in transforming raw tokens into something a neural network ca=
n work with. Each token ID from the vocabulary is mapped to a dense vector =
of fixed size, known as an embedding vector. Instead of representing words as sparse one-hot encodings, embeddings ca=
pture semantic meaning, so that related words like king and queen end up closer together in vector space t=
han unrelated words like king and ba=
nana.This embedding matrix has dimensions (vocabulary size =C3=97 embedding dimension), where the embedding=
 dimension is a hyperparameter you choose. The dimensions of the positional=
 encoding matrix should match those of the embedding matrix because, as you=
 can see in the architecture, the embedding matrix is enriched with informa=
tion by summing up with the positional encoding matrix element-wise.Press enter or click to view image in full siz=
e=
Image By AuthorFor simp=
licity, assume that the embedding dimension and attention dimension are the=
 same for output. For input, the embedding layer represents the whole vocab=
ulary of words. If the model is trained on 10000 unique words (tokens liter=
ally), that will be the input shape. For GPT-2, the standard is 50257 words=
. Similarly, the input for the positional encoding layer will be equal to t=
he number of tokens the input layer of the LLM has (number of tokens the LL=
M processes at one go). Call it the context length.self.embedding =3D torch.nn.Embedding(vocab_size, attention_dim)self.positional_embedding =3D torch.nn.Embedding(context_length, attenti=
on_dim)In the forward method, define the flow as.embedd=
ings =3D self.embedding(context)context_len =3D context.shape[1]position =3D torch.arange(context_len, devi=
ce=3Dcontext.device).unsqueeze(0)pos=
ition_embeddings =3D self.positional_embedding(position)e =3D embed=
dings + position_embeddingsYou will see it in acti=
on in the full GPT decoder that we will build soon.



### Self-Attention: How Tokens Gossi=
p About Each Other
Now, let us code the self-attention modu=
le using what we discussed earlier in the section. Define the trainable query, key, and value matrices.self.w_key =3D torch.nn.Linear(embed_dim, attention_dim, bias=3Dbia=
s)self.w_query =3D torch.nn.Linear(emb=
ed_dim, attention_dim, bias=3Dbias)self.w_val=
ue =3D torch.nn.Linear(embed_dim, attention_dim, bias=3Dbias)=
Compute the query, key, and value vectors.k =3D self.w_key(x)=
   # (B, T, A)q =3D self.w_query(x)=
 # (B, T, A)v =3D self.w_value(x) # (B, T, A)```where,B: batch size,T: context length,A: attention =
dimension```Now, compute the attention =
scores using the product of the query and key vectors. Take the transpose o=
f the key vector to match dimensions. Also, before applying softmax, normal=
ize the value by dividing it by the square root of the embedding dimension.=
Image By Authorscores =3D (q @ k.transpose(-2, -1)) / (k.size(-1) ** 0.5)  # (B, T, T)Mask the futur=
e positions and apply softmax activation. Finally, return the product of at=
tention weights and value vector.mask =3D torch.triu(torch.ones(T, T, device=3Dx=
.device), diagonal=3D1).bool()scores =3D scores.masked_fill(mask, float('-1e10'))attn =3D scores.softmax(dim=3D-1)  # (B, T, T)final =3D attn @ =
v  # (B, T, A)=
Don't forget to add dropouts as needed to stabilize the training. It masks =
additional attention weights randomly according to the percentage you speci=
fy. Here it is 0.1, meaning 10% of the weights are masked randomly.Drop=
out is considered for the entire matrix, not just the remaining values.=
Press enter or click to view image in full =
sizeImage By AuthorPutting=
 it all together as a module,class SelfAttention(torch.nn.Module):    def __in=
it__(self, embed_dim, attention_dim, bia=
s=3DFalse, dropout=3D0.1):        supe=
r().__init__()        self.w_key =3D torch.nn.Linear(embed_dim, =
attention_dim, bias=3Dbias)        self.w_query =3D torch.nn.Linear(emb=
ed_dim, attention_dim, bias=3Dbias)        self.w_value =3D torch.nn.Li=
near(embed_dim, attention_dim, bias=3Dbias)        self.dropout =3D tor=
ch.nn.Dropout(dropout)    def forward(self, x):        B, T, _ =3D x.size()        k =3D se=
lf.w_key(x)   # (B, T, A)        q =
=3D self.w_query(x) # (B, T, A)    =
    v =3D self.w_value(x) # (B, T, A)        # Scaled dot-product attention        scores =3D (q @ k.transpose(-2=
, -1)) / (k.size(-1) ** 0.5)  # (B, T, T)        # Causal mask (future positions masked)        mask =3D=
 torch.triu(torch.ones(T, T, device=3Dx.device), diagonal=3D1).bool()     =
   scores =3D scores.masked_fill(mask, float('-1e10'))        attn =3D=
 scores.softmax(dim=3D-1)  # (B, T, T)        attn =3D self.dropout(a=
ttn)        return attn @ v  # (B, T, A)Image By Author=



### Multi-Head A=
ttention: The Group Chat in Your Model=E2=80=99s Brain
You =
have a self-attention module ready. But for a massive model that is being t=
rained on massive datasets, this won=E2=80=99t be enough. Thus, we combine =
multiple attention heads in parallel to create multi-head attention. The co=
re idea is that multiple attention heads can focus on different aspects of =
the data, capturing relationships across various subspaces and positions, w=
hich allows the model to learn richer and more complex patterns.class MultiHeadAttenti=
on(torch.nn.Module):    def =
__init__(self, num_heads, embed_dim, attention_dim, dropout=3D0.1):        sup=
er().__init__()        self.head_size =3D attention_dim//num_hea=
ds        self.heads =3D torch.nn.ModuleList()        for i in range(num_heads):            self.heads=
.append(SelfAttention(embed_dim=3Dembed_dim, attention_dim=3Dself.head_size=
,dropout=3Ddropout))    def forward(self,x):        head_outputs =3D []        for head in self.he=
ads:            head_outputs.append(head(x)) #B x T x A//num_heads        concatenated =3D torch.cat(head_=
outputs, dim =3D 2)        return concatenated=
Image By AuthorThe total attention dimension is divided among the heads, en=
abling parallel processing. The resulting attention matrix outputs are conc=
atenated to get a full result.



### Feed-Forward Networks
After attention=
, each token embedding is passed through a small feed-forward network. This=
 could simply be two linear layers, which consist of an up-projection and a=
 down-projection with a non-linear activation in between. You can make it m=
ore complex according to your architecture.class FeedForward(torch.nn.Module):    def __init__(self,attention_dim):        super().__init__()=
        self.up =3D torch.nn.Linear(attention_dim,attention_dim*4)        self.gelu =3D torch.nn.GELU()    =
    self.down =3D torch.nn.Linear(attention_dim*4,attention_dim)    def forward(self,x):        return self=
.down(self.gelu(self.up(x)))Image By Author



In [ ]:
class FeedForward(torch.nn.Module):    def __init__(self,attention_dim):        super().__init__()=
        self.up =3D torch.nn.Linear(attention_dim,attention_dim*4)        self.gelu =3D torch.nn.GELU()    =
    self.down =3D torch.nn.Linear(attention_dim*4,attention_dim)    def forward(self,x):        return self=
.down(self.gelu(self.up(x)))


### The Decoder with Residual =
Connections
Now that you have the multi-head attention modu=
le and the feed-forward network, let us build the decoder part of the model=
.Press enter or click to view image in =
full size=
SourceClosely observe the diagram here. You=E2=80=99ll notice that =
each sub-layer (attention or feed-forward) doesn=E2=80=99t simply replace i=
ts input. Instead, the original input is added back=
 to the output of the sub-layer. This shortcut is called a residual connection. It helps the model train mor=
e effectively by preserving the original signal and avoiding issues like va=
nishing gradients.On top of this, every residual connection=
 is followed by Layer Normalization. Unlik=
e BatchNorm, which normalizes across a bat=
ch, LayerNorm normalizes across the features of=
 a single token embedding, centering the activations of a neural network la=
yer around a mean of 0 and normalizing their variance to 1. This ensures th=
at the scale and distribution of activations remain stable, which is critic=
al in deep networks like Transformers, where stacking many layers can other=
wise destabilize training.Stack up the layers as in the dia=
gram, and you will have this.class Decoder(torch.nn.Module):    def __init__(self,num_heads,embed_dim,attention_dim, d=
ropout=3D0.1):        super().__init__()        self.masked_mul=
tihead =3D MultiHeadAttention(num_heads, embed_dim, attention_dim, dropout)=
        self.feed_forward =3D FeedForward(attention_dim)        sel=
f.n1 =3D torch.nn.LayerNorm(attention_dim)        self.n2 =3D torch.nn.=
LayerNorm(attention_dim)    def forward(self,x):        e =3D self.masked_multihead(self.n1(x))    =
    e =3D  e + x        e =3D self.feed_forward(self.n2(e))        =
return eImage By AuthorPutting It Al=
l Together: The Transformer SkeletonPerfect. Now, you just=
 need to design the input and output parts of the model. The input part wil=
l be the embedding and positional encoding layer, which we created earlier. They convert raw token IDs into dense vectors e=
nriched with positional information, preparing them for the Transformer blo=
cks.The output part gets a little more interesting. You lea=
rned that the LLM output must represent a probability distribution over the=
 entire vocabulary. That means, every token we expect it to learn should ha=
ve its own output, or in other words, the output si=
ze equals the vocabulary size. So we project them linearly through=
 a layer of the vocabulary size, which is also called the LM (language modelling) head. The LM head is just a linear =
projection without activation. The output is still not probabilities, but c=
an be called logits. Softmax is applied afterward during train=
ing/inference to get the probability distribution.i=
mport torchfrom torch import nnclass GPT(nn.Module):  =
  def __init__(self, num_heads, vocab_size,=
 embed_dim, attention_dim, num_blocks, context_length, dropout_rate)=
:        super().__init__()   =
     self.embedding =3D nn.Embedding(vocab_size, attention_dim)        =
self.positional_embedding =3D nn.Embedding(context_length, attention_dim)        self.decoders =3D nn.ModuleList([            Decoder(num_=
heads, attention_dim, attention_dim, dropout_rate) for _ in range(num_blocks)        ])        self.exit=
_norm =3D nn.LayerNorm(attention_dim)        self.linear =3D nn.Linear(=
attention_dim, vocab_size)    def forward(self, context):        embeddings =3D self.embedding(cont=
ext)        context_len =3D context.shape[1=
]        position =3D torch.arange(context_len, device=3Dcontext=
.device).unsqueeze(0)        positio=
n_embeddings =3D self.positional_embedding(position)        e =3D e=
mbeddings + position_embeddings        for decoder in self.decoders:            e =3D decoder(e)        =
return self.linear(self.exit_norm(e))Multip=
le decoders are added according to the scale you require. For large GPT mod=
els, this can go up to 25 heads, which may seem easy, but will consume a to=
n of memory and require massive data to train.Image By AuthorSo far, so good. You can test your model using a simple sequence gene=
rating function.def top_k_logits(logits, k)=
:    v, ix =3D torch.topk(logits, =
k)    out =3D logits.clone()  =
  out[out &lt; v[:, [-1]]] =3D float('-inf'=
)    return outdef generate(model, max_new_tokens, context, context_length, temperature=3D1.0, top_k=3DNone):    res =3D []    for _ =
in range(max_new_tokens):        i=
f context.shape[1] &gt; context_length:            context =3D context[:, -context_length:]        logi=
ts =3D model(context)  # [B, T, V]=
        logits =3D logits[:, -1, :]  # [=
B, V]        logits =3D logits / max(temperature, 1e-3)        if top_k is not None:            logits=
 =3D top_k_logits(logits, top_k)        if torch.isnan(logits).any() or torch.isinf(logits).any():     =
       raise ValueError("Logits contain NaN or Inf")        probabilit=
ies =3D nn.functional.softmax(logits, =
dim=3D-1)        probabilities =3D t=
orch.clamp(probabilities, min=3D1e-9, max=
=3D1.0)        next_token =3D to=
rch.multinomial(probabilities, 1)  # [B, 1]=
        context =3D torch.cat((con=
text, next_token), dim=3D1) =
   return contextstart_context =3D "I want something"model =3D GP=
T(num_heads,vocab_size,embed_dim,attention_dim,num_blocks,context_le=
ngth, dropout_rate).to(device)mode=
l.eval()token_ids =3D generate(    model=3Dmodel,    context=3D=
text_to_token_ids(start_context, token=
izer, device),    max_new_tokens=3D10,    context_length=3Dcontext_length)print("Output text:\n", token_ids_to_text(token_ids, tokenizer))If things went right, you can see some random garbag=
e output like this.Output text: I=
 want something introduce=E3=82=A6 coaches Kard Judais=
m trendsCommerce rotating i=
nfiltration approach=



## Model =
Pretraining
The model pretraining is, in simple words, inte=
nded to enable the model to understand and speak basic English. The model m=
ust be able to generate a grammatically correct sequence of words that make=
s some sense, although there may not be much context. For this, we need a d=
ataset that can provide a large corpus of English text. You can choose from=
 numerous publicly available datasets, such as IMDb.



### Training
Before training, you need to initialize the =
weights to ensure your model starts training from a predefined point. This =
is the standard practice for GPT models.def =
initialize_weights(module):    if isinstance(module, nn.Linear):=
        torch.nn.init.normal_(module.weight, mean=3D0.0, std=3D0.02)        =
if module.bias is not None:            torch.nn.init.zeros_(module.bias) =
   elif i=
sinstance(module, nn.Embedding):        torch.nn.init.normal_(mo=
dule.weight, mean=3D0.0, std=3D0.02)    elif isinstance(module, nn.LayerNorm):=
        torch.nn.init.ones_(module.weight)        torch.nn.init.zer=
os_(module.bias)model.apply(initialize_weights)Let us start with the loss function. LLMs are, in a way, performing =
multi-class classification, where each possible word in the vocabulary is a=
 class, and the model outputs a probability distribution over all words. Th=
us, we use the cross-entropy loss.Press enter or click to view image in full size=
Image By AuthorCross-en=
tropy rewards high probability for the right word, and punishes when the co=
rrect word has low probability.Consider our previous exampl=
e.Index: Token                            0 =E2=86=92 "The"           Inputs: ["The", "cat", "sat", "on", "the"]1 =E2=86=92 "cat"           Targets: ["cat", "sat", "on", "the",=
 "mat"]  2 =E2=86=92 "sat"3 =E2=86=92 "on"4 =E2=86=92 "the"5 =E2=86=
=92 "mat"Press enter or click to view image in full sizeImage By AuthorComputing loss for each p=
rediction,Position 0: Targe=
t =3D "cat" =E2=86=92 P =3D 0.90 =
=E2=86=92 L0 =3D =E2=88=92log=E2=81=A1(0.9) =E2=89=88 0.105Position 1: Target =3D "sat" =E2=86=92 P =3D 0.10 =E2=86=92 L1 =3D =E2=88=92log=E2=81=A1(0=
.1) =3D 2.302Position 2: Target =3D "on" =E2=86=92 P =3D 0.05 =E2=86=92 L2 =
=3D =E2=88=92log=E2=81=A1(0.05) =3D 2.996Position =
3: Target =3D "the" =E2=86=
=92 P =3D 0.75 =E2=86=92 L3 =3D =E2=88=92log=E2=81=A1(0.75) =E2=89=88 0.288=
Position 4: Target =3D "mat" =E2=86=92 P =3D 0.75 =E2=86=92 L4 =3D =E2=88=92=
log=E2=81=A1(0.75) =E2=89=88 0.288Taking the average =
loss,Lavg =3D (0.105+2.302+2.996+0.28=
8+0.288) =E2=80=8B/ 5 =E2=80=8B=E2=89=88 1.20Since wrong pr=
edictions give huge loss values, a few bad predictions dominate the average=
 loss. Using this loss, define evaluation functions. We have explicitly men=
tioned @torch.no_grad() because the=
 model weights shouldn=E2=80=99t be updated while the loss is being calcula=
ted.c=
riterion =3D nn.CrossEntropyLoss()def calc_loss_batch(input_batch, target_batch, model, device):   =
 input_batch =3D input_batch.to(device, non_blocking=3DTrue)    target_batch =3D target_batch.to(device, non_b=
locking=3DTrue)    logits =3D m=
odel(input_batch)  # [B, T, V]    B=
, T, V =3D logits.shape    loss =3D criterion(logits.view(B * T, V), ta=
rget_batch.view(B * T))    return l=
oss@torch.no_grad()def calc_l=
oss_loader(data_loader, model, device, n=
um_batches=3DNone):    if len(=
data_loader) =3D=3D 0:        return float("nan")    model.eval()    total_loss =3D 0.0    num_batches =3D len=
(data_loader) if num_batches is None else mi=
n(num_batches, len(data_loader)=
)    for i, (inp, tgt) in enumerate(data_loader):        if i &gt;=
=3D num_batches:            break        loss =3D calc_loss_batch(inp, tgt, model, device)        tota=
l_loss +=3D loss.item()    model.train()    return total_loss / num_batches@torch.no_grad()def evaluate_model(model, train_loader, val_loader, device, eval_iter=3D1):    train_loss =3D calc_loss_loader(t=
rain_loader, model, device, num_batches=3Deval_iter)    val_loss   =3D =
calc_loss_loader(val_loader, model, device, num_batches=3Deval_iter)   =
 return train_loss, val_lossNote that you have explicitly set model.eval() while calculating loss, set it back to model.train() once done. When it comes to=
 training, the biggest challenge is how to adjust the learning rate over ti=
me. If it=E2=80=99s too high, the model won=E2=80=99t converge. If it=E2=80=
=99s too low, training will be painfully slow. To balance this, we often us=
e a scheduler that adapts the learning rate during training.In this setup, we use a custom scheduler called CosineWithWarmup.class CosineWithWarmup(torch.optim.lr_scheduler.=
_LRScheduler):    def __init__(self, =
optimizer, warmup_steps, total_steps, base_lr, min_lr, last_epoch=3D-1):        self.warmup_steps =3D max(1=
, warmup_steps)        self.total_steps =3D max(self.warmup_steps + 1, tot=
al_steps)        self.base_lr =3D base_lr        self.min_lr =3D mi=
n_lr        super().__init__(optim=
izer, last_epoch)    def get_lr(sel=
f):        step =3D self.last_epoch + 1        lrs =3D []        fo=
r _ in self.base_lrs:       =
     if step &lt;=3D self.warmup_steps:=
                lr =3D self.base_lr * step / self.warmup_steps     =
       else:                progres=
s =3D (step - self.warmup_steps) / max=
(1, self.total_steps - self.warmup_steps=
)                lr =3D self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))            lrs.append(lr)        =
return lrsWarmup phase: For the first few steps, the lear=
ning rate linearly ramps up from 0 to the base value. This helps stabilize =
training, especially for large models like GPT, which can otherwise diverge=
 early on.Cosine decay: After warmup, the=
 learning rate gradually decreases following a cosine curve, smoothly decay=
ing towards a minimum value (min_lr=
). This prevents sudden drops and helps the model =E2=80=9Csettle=E2=80=9D =
into a good local minimum.Start small =E2=86=92 Rise steadily =E2=86=92 Decay smoothly.=
settings =3D {    "learning_ra=
te": 3e-4,              "weight_decay": 0.1,            # Standard for GPT-style training    "num_epochs": 300,                "batch_size": 32,               # Balance for GPU memory vs convergence    "warmup_steps": 1500,           # Warmup helps avoid divergence early    "max_lr": 3e-4, =
                    "min_lr": 3e-5,                     "eval_freq": 200,                   "eval_iter": 20,                    "gradient_clip":=
 1.0,               "patience": 50,                     "min_improvement": 1e-4,    "print_interval": 1,       =
         "generate_interval": 5     =
     }train_dataloa=
der =3D create_encoded_dataloader(    train_text_data,    toke=
nizer=3Dtokenizer,    batch_siz=
e=3Dsettings["batch_size"],    max_length=3Dcontext_length,    stride=3Dcontext_length,    shuffle=3DTrue,    drop_last=3DTrue)test_dataloader =3D create_encoded_dataloader(    test_text_data,    tokenizer=3Dtokenizer,    batch_size=3Dsettings["batch_size"],    max_length=3Dcontext_length,    stride=
=3Dcontext_length,    shuffle=
=3DFalse,    drop_last=3DTrue)It'=
s a convenient practice to manage your settings in one block. The next piec=
e is the training loop, where the actual learning happens.def train_model(    model,    train_loader,    val_lo=
ader,    device,    settings,    save_path=3D"checkpoints/gpt_256_256_8_8.pt",):    =
torch.manual_seed(123)    if torch.cuda.is_available():        torch.cu=
da.manual_seed_all(123)    model=
.to(device)    optimizer =3D torch.optim.AdamW(        model.pa=
rameters(),        lr=3Dsettings["learning_=
rate"],        weight_decay=3Dsettings["weight_decay"],        betas=3D(=
0.9, 0.95),    )    t=
otal_steps =3D settings["num_epochs"] * =
len(train_loader)    scheduler =3D=
 CosineWithWarmup(        optimizer,        warmup_steps=3Dsettings=
["warmup_steps"],        total_steps=
=3Dtotal_steps,        base_lr=3Dsettings["=
max_lr"],        min_lr=3Dsettings["=
min_lr"],    )    train_losses, val_losses, tokens_seen_=
track =3D [], [], []    tokens_seen, global_step =3D 0, -1    best_val_l=
oss, patience_counter =3D float("inf"), 0    for epoch in range(settings["num_epochs"]):        model.train()          for step, (inp, tgt) in enumerate(train_loader):            loss =3D calc_loss_batch(inp, tgt, model,=
 device)            loss.backward()            # gradient clipping            torch.nn.utils.clip_=
grad_norm_(model.parameters(), settings["gradie=
nt_clip"])            optimizer.step()            optimi=
zer.zero_grad(set_to_none=3DTrue)  =
          scheduler.step()            global_step +=3D 1            tokens_seen +=3D inp.numel()   =
         # evaluation            if global_step % settings["eval_freq"] =3D=3D 0:                train_loss, val_loss =3D evaluate_model(         =
           model, train_loader, val_loader, device,                    =
eval_iter=3Dsettings["eval_iter"],  =
              )                train_losses.append(train_loss)     =
           val_losses.append(val_loss)                tokens_seen_track=
.append(tokens_seen)                lr_now =3D optimizer.param_groups[0]["lr"]                print(f"Ep {epoch+1} | step {global_st=
ep:06d} | lr {lr_now:.3e} "                      f"| train {train_loss:.3f} | val {val_l=
oss:.3f}")        =
        # early stopping           =
     if val_loss + settings["min_improvement"] &lt; best_val_loss:         =
           best_val_loss =3D val_loss                    patience_count=
er =3D 0                    os.maked=
irs(os.path.dirname(save_path) or ".", exist_ok=3D=
True)                    torch.save({                       =
 "model_state": model.state_dict(), =
                       "optimizer_state"=
: optimizer.state_dict(),                        "epoch": epoch,                        "global_step": global_step,                    }, save_=
path)                    print(f"[Checkpoint saved at step {global_step}]")                else:                    patience_counter +=3D 1                    if patience_counter &gt;=3D settings["patience"]:                        print("Early stopping triggered.")                        return train_losses, val_losses, tokens_seen_track    return train_losses, val_losses, tokens_seen_track=
There are three things worth mentioning in the code. Thes=
e are small details, but they make a big difference in training stability a=
nd efficiency.Gradient ClippingDuring backpropagation, the model calculates gradients for each par=
ameter. In practice, sometimes these gradients can become extremely large (=
exploding gradients), especially in deep n=
etworks or when training on long sequences. If that happens, the weight upd=
ates can destabilize training and cause the loss to diverge. Gradient clipp=
ing prevents this by putting a ceiling on the size of the gradients.Early StoppingTraining LLMs takes days. So i=
t's necessary to keep a check on the compute and cut off if there is no mea=
ningful improvement.AdamWFor the actu=
al weight updates, we use AdamW, a =
modern optimizer that combines the benefits of Adam with proper weight decay regularization. Adam adapts the learning=
 rate individually for each parameter, which helps models converge faster. =
The =E2=80=9CW=E2=80=9D stands for decoupled weight decay. Unlike classic Adam, it cleanly separates weight decay from gradient =
updates, which improves generalization.This is some r=
andom output I got.the movie starts slow and i thought it was going to be boring=
, but then going to be interesting. the acting is okay, some are boring=
 felt like they just gave up. still, it was not the worst film i=E2=80=
=99ve seen



In [ ]:
the movie starts slow and i thought it was going to be boring=
, but then going to be interesting. the acting is okay, some are boring=
 felt like they just gave up. still, it was not the worst film i=E2=80=
=99ve seen


In [ ]:
def train_model(    model,    train_loader,    val_lo=
ader,    device,    settings,    save_path=3D"checkpoints/gpt_256_256_8_8.pt",):    =
torch.manual_seed(123)    if torch.cuda.is_available():        torch.cu=
da.manual_seed_all(123)    model=
.to(device)    optimizer =3D torch.optim.AdamW(        model.pa=
rameters(),        lr=3Dsettings["learning_=
rate"],        weight_decay=3Dsettings["weight_decay"],        betas=3D(=
0.9, 0.95),    )    t=
otal_steps =3D settings["num_epochs"] * =
len(train_loader)    scheduler =3D=
 CosineWithWarmup(        optimizer,        warmup_steps=3Dsettings=
["warmup_steps"],        total_steps=
=3Dtotal_steps,        base_lr=3Dsettings["=
max_lr"],        min_lr=3Dsettings["=
min_lr"],    )    train_losses, val_losses, tokens_seen_=
track =3D [], [], []    tokens_seen, global_step =3D 0, -1    best_val_l=
oss, patience_counter =3D float("inf"), 0    for epoch in range(settings["num_epochs"]):        model.train()          for step, (inp, tgt) in enumerate(train_loader):            loss =3D calc_loss_batch(inp, tgt, model,=
 device)            loss.backward()            # gradient clipping            torch.nn.utils.clip_=
grad_norm_(model.parameters(), settings["gradie=
nt_clip"])            optimizer.step()            optimi=
zer.zero_grad(set_to_none=3DTrue)  =
          scheduler.step()            global_step +=3D 1            tokens_seen +=3D inp.numel()   =
         # evaluation            if global_step % settings["eval_freq"] =3D=3D 0:                train_loss, val_loss =3D evaluate_model(         =
           model, train_loader, val_loader, device,                    =
eval_iter=3Dsettings["eval_iter"],  =
              )                train_losses.append(train_loss)     =
           val_losses.append(val_loss)                tokens_seen_track=
.append(tokens_seen)                lr_now =3D optimizer.param_groups[0]["lr"]                print(f"Ep {epoch+1} | step {global_st=
ep:06d} | lr {lr_now:.3e} "                      f"| train {train_loss:.3f} | val {val_l=
oss:.3f}")        =
        # early stopping           =
     if val_loss + settings["min_improvement"] &lt; best_val_loss:         =
           best_val_loss =3D val_loss                    patience_count=
er =3D 0                    os.maked=
irs(os.path.dirname(save_path) or ".", exist_ok=3D=
True)                    torch.save({                       =
 "model_state": model.state_dict(), =
                       "optimizer_state"=
: optimizer.state_dict(),                        "epoch": epoch,                        "global_step": global_step,                    }, save_=
path)                    print(f"[Checkpoint saved at step {global_step}]")                else:                    patience_counter +=3D 1                    if patience_counter &gt;=3D settings["patience"]:                        print("Early stopping triggered.")                        return train_losses, val_losses, tokens_seen_track    return train_losses, val_losses, tokens_seen_track=


In [ ]:
settings =3D {    "learning_ra=
te": 3e-4,              "weight_decay": 0.1,            # Standard for GPT-style training    "num_epochs": 300,                "batch_size": 32,               # Balance for GPU memory vs convergence    "warmup_steps": 1500,           # Warmup helps avoid divergence early    "max_lr": 3e-4, =
                    "min_lr": 3e-5,                     "eval_freq": 200,                   "eval_iter": 20,                    "gradient_clip":=
 1.0,               "patience": 50,                     "min_improvement": 1e-4,    "print_interval": 1,       =
         "generate_interval": 5     =
     }train_dataloa=
der =3D create_encoded_dataloader(    train_text_data,    toke=
nizer=3Dtokenizer,    batch_siz=
e=3Dsettings["batch_size"],    max_length=3Dcontext_length,    stride=3Dcontext_length,    shuffle=3DTrue,    drop_last=3DTrue)test_dataloader =3D create_encoded_dataloader(    test_text_data,    tokenizer=3Dtokenizer,    batch_size=3Dsettings["batch_size"],    max_length=3Dcontext_length,    stride=
=3Dcontext_length,    shuffle=
=3DFalse,    drop_last=3DTrue)


In [ ]:
Index: Token                            0 =E2=86=92 "The"           Inputs: ["The", "cat", "sat", "on", "the"]1 =E2=86=92 "cat"           Targets: ["cat", "sat", "on", "the",=
 "mat"]  2 =E2=86=92 "sat"3 =E2=86=92 "on"4 =E2=86=92 "the"5 =E2=86=
=92 "mat"


## Teaching Your Model to Follow (and Sing Coldplay)
Congratulations! After some long waiting, you have your own LLM capable o=
f speaking basic English (although it may not make much logical sense as we=
 trained on IMDb and the architecture is small and basic). Now, time to tea=
ch it Coldplay style patterns. To do this, we fine-tune the model on a smal=
l Coldplay dataset. This is actually how large language models like GPT are=
 built:First, they undergo pretraini=
ng on a massive general-purpose dataset (billions of tokens from b=
ooks, websites). This teaches them grammar, vocabulary, and general world k=
nowledge.Then, they are fine-tuned on sma=
ller, specialized datasets to adapt them to a particular style or task (cha=
tting, coding, medical Q&amp;A, legal reasoning, etc.). Without fine-tuning=
, GPT would just be a giant text predictor. With fine-tuning, it becomes co=
nversational, safe, and aligned to the tasks we actually care about.You can use this dataset and use the same preprocessing steps to get it rea=
dy. Use the same training loop for fine-tuning, but tweak the settings a bi=
t.set=
tings_ft =3D {    "learning_rate": 1=
e-5,          # Lower LR for fine-tuning to pr=
eserve pretrained weights    "weight=
_decay": 0.01,           # Reduced weig=
ht decay    "num_epochs": 5, =
               # Fewer epochs since Coldplay d=
ataset is small    "batch_size": 4,                # Smaller batch size for=
 small dataset    "warmup_steps": 100,            # Shorter warmup    "max_lr": 1e-5,    "min_lr": 1e-6,    =
"eval_freq": 50,                   "=
eval_iter": 5,                     "=
gradient_clip": 0.5,           # gentle=
r clipping    "patience": 3, =
                     "min_improvement": 1e-4,    "print_interval": 1,    "generate_interval": 2}=
train_losses_ft, val_losses_ft, tokens_seen_ft =3D train_model(    mode=
l,    train_dataloader_ft,    val_dataloader_ft,    tokenizer,    device,    settings=3Dsettings_ft,    context_length=3Dconte=
xt_length,    save_path=3D"checkpoints/gpt_=
512_512_8_8_finetuned_coldplay.pt",    sample_prompt=3D"Look at the star look how the "  )Let=E2=80=99s check the output once again,lights go out and the s=
tars begin to fall i hear your voice across the night  lights are runni=
ng in circles chasing the echoes  you are the star that keeps me alive =
 Oh-ooh-oh-ooh oh, oh  i will follow you, i will follow you



## Wrappin=
g Up
And that=E2=80=99s a wrap! You have built your own tra=
nsformer architecture from scratch and trained it with nothing but pure PyT=
orch. You have seen everything you need from concepts to code, with more in=
tricate information explained that you won=E2=80=99t find together anywhere=
 else. Here is the complete notebook.Of course, this doesn=E2=80=99t end here. Modern AI has a lot of =
advanced architectures and algorithms built on top of this to enable the as=
tonishing capabilities we see today. I will definitely be sharing more deep=
 dives in the coming days. It=E2=80=99s fine if you feel these concepts a b=
it complex to understand in one go. My goal was to distill the entire learn=
ing process I went through into one place, so you can focus on building rather than searching the web. Drop some claps and =
follow if you liked it (I am sure you did!).=F0=9F=9A=
=A9Explore reasoning LLMs in my latest write-up:



## No Libraries No Shortcuts: Reasoning Models from Scratch wi=
th PyTorch =E2=80=94 Part 1


### The no BS Guide to implementing LLMs with Mixture =
of Experts, RoPE, and Grouped Query Attention from scratch
pub.towardsai.=
net



## References
[1] Ashish Vaswani, Noam Sh=
azeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Ka=
iser, Illia Polosukhin(2017).Attention is =
all you need.arXiv:1706.03762[2] Ilya Sutskever, Tim Sa=
limans, Karthik Narasimhan, Alec Radford(2018).Improving=
 Language Understanding by Generative Pre-Training[3] J=
immy Lei Ba, Jamie Ryan Kiros, Geoffrey E. Hinton(2016).Layer Normalization.arXiv:1607.06450[4] R=
obin M. Schmidt(2019). Recurrent Neural Ne=
tworks (RNNs): A gentle Introduction and Overview.arXiv:1912.05911[5] Sebastian Raschka.LLMs From Scratch



## Images
If not otherwise stated, all images ar=
e created by the author.Large Language Models=
Artificial IntelligenceMachine LearningDeep Lear=
ningData Science=
=
1.3K1.3K17=
=
Follow



## Publi=
shed in Towards AI
118K foll=
owers=C2=B7Last published&nbsp;just nowWe build Enterprise AI. We teach what =
we learn. Join 100K+ AI practitioners on Towards AI Academy. Free: 6-day Ag=
entic AI Engineering Email Guide: https://email-course.towardsai.net/=
FollowFollow=



## Written by Ashish Abraham
=
916 follo=
wers=C2=B733 fol=
lowing@SnapAR | AI Engineer &amp; StoryTeller=E2=9C=A8Follow=



## Responses (17)
Soub=
hikCancelRespondAmaresh AdakOct 4, 2025=
This is exactly the kind of deep dive I've been looking for! I remember the=
 first time I tried to understand attention mechanisms from the original pa=
per =E2=80=94 it felt like deciphering ancient scrolls. What really resonat=
es with me here is how you've=E2=80=A6more65Reply=
Jaron FralickOct 6, 2025=
Oh my gosh, thank you!!!! This is exactly what I've been looking =
for! A complete guide like this is invaluable for those of us that wany to =
build our own models and not have to rely on models with unknown biases and=
 uncessary fluff packed in!14Replymohamad shahkhajehOct 5, 2025The article nicely highlights that all GPT mo=
dels, despite refinements, still fundamentally rely on the original Transfo=
rmer architecture.11ReplySee all responses



In [ ]:
The article nicely highlights that all GPT mo=
dels, despite refinements, still fundamentally rely on the original Transfo=
rmer architecture.


In [ ]:
Oh my gosh, thank you!!!! This is exactly what I've been looking =
for! A complete guide like this is invaluable for those of us that wany to =
build our own models and not have to rely on models with unknown biases and=
 uncessary fluff packed in!


## More from Ashish Abrah=
am and Towards AI
InTowards AI=
by=
Ashish Abraham=



## No Libraries No Shortcuts: Reasoning LLMs from Scratch with PyTorch =E2=
=80=94 Part 1


### The no BS Guide to implementing LLMs with Mixture of Expe=
rts, RoPE, and Grouped Query Attention from scratch
=
Jan 1A clap icon321InTowards AIbyAdham Khaled



## I=E2=80=99ve Been Recommendin=
g DeepSeek &amp; Kimi for Months. Then Anthropic Published This.
=



### A brea=
kdown of the most explosive AI security report of 2026 =E2=80=94 and what i=
t honestly means for everyone using Chinese AI tools.
Feb 25A clap icon1.7KA response icon=
85=
=
InT=
owards AIbyRick Hightower



## Claude Code Agent Skills 2.0: From Custom Instr=
uctions to Programmable Agents


### Skills are no longer instructions. They =
are programs.
Mar 9A clap icon382A response icon6=
=
InT=
owards AIbyAshish Abraham



## No Libraries No Shortcuts: =
Reasoning LLMs from Scratch with PyTorch=E2=80=8A=E2=80=94=E2=80=8APart 2The no BS Guide to implementing reasoning models from scratch with SFT &=
amp; RL=
Jan 20A clap icon231A response icon1See all from Ashish AbrahamSee all from Towards AI=
Recommended from Medium=
InArtificial CornerbyThe PyCoach



## The Best AI Tools for 20=
26


### If you=E2=80=99re going to learn a new AI tool, make sure it=E2=80=
=99s one of these
=
Dec 1, 2025A clap icon=
5.4KA response icon3=
21=
InTowards AIbyDivy Yadav=



## =
9 RAG Architectures Every AI Developer Must Know: A Complete Guide with Exa=
mples


### Architectures beyond Naive Rag to build reliable production AI Sy=
stems
=
Dec 19, 2025A clap icon2.4KA response icon43=
=
InL=
evel Up CodingbyFareed Khan



## Building a Self-I=
mproving Agentic RAG System


### Specialist agents, multi-dimensional eval, =
Pareto front and more.
=
Nov 15, 2025A clap icon1.8KA response =
icon18=
=
=
Michal M=
alewicz



## The End of Dashboards and Design Systems


### Design is b=
ecoming quietly human again.
Nov 26, =
2025A clap icon6=
.5KA =
response icon261=
InWomen in TechnologybyAlina Kovtun=E2=9C=A8



## Stop Memorizing Design Patterns: Use This D=
ecision Tree Instead


### Choose design patterns based on pain points: apply=
 the right pattern with minimal over-engineering in any OO language.
Jan 29A clap=
 icon7KA response icon68=
=
InData Science CollectivebyMarina Wyss



## Should You Still Learn to Code in 2026?


### The answer isn=
=E2=80=99t as obvious as I used to believe.
Feb 23A clap icon2KA response icon77=
See more =
recommendationsHelpStatusAboutCareersPressBlogPrivacyRulesTermsText to=
 speech=

=

------MultipartBoundary--eFfsVS1Hba2ZnY6Mm8aqBwcs4D5PLyLFassH9VwgLy----
Content-Type: image/webp
Content-Transfer-Encoding: base64
Content-Location: https://miro.medium.com/v2/resize:fit:500/format:webp/1*DkmqJXLDpX0tEnn53OoShw.png

UklGRvQIAABXRUJQVlA4WAoAAAAIAAAA+QAAeQAAVlA4IBQIAACwLQCdASr6AHoAPnU0l0mkoqIh
I1FKAJAOiWlu4XVRG/N78h/0Dtc/s38a/YDqoO/fsXoE/xf61/bP65+5Pql3m8AL8O/jn+Q/qP7i
8CuAD8v/mf+Y/tH7jf4DnR+tfsAfyL+d/6n+u+x39z8AbzX2AP5f/cf+f7K39B/yP7j+WPtT/Nf7
x/2/8b8BH8r/pH+1/uH72f5v5nvXr+1vsj/smPa7u7u7u7u7u7u7u64q0wtziry86YyfnGXjwM2n
PlWsBEREREOqzuTzFAJkG1ldFvyMxyFoundcmYJPt4cC2C2G0Z6AzxYF4U/g61uhxZigt8efD8ks
qKQJp9qpZ/61g0TzuO3n61G8iId/Udjs09T/TgnrCvR3/Sx9MgGstYA8ZkyPw/wHS9ZvWoqgYh9J
jkcGS3ibIKTUZlrBYkReIeVuqqqqqxLjdN7cm7BHc3gwF70IZb2EZNVjWEeBb0zMzMzMoN72UmUv
Fj3lwRERERERHD23AAD+/7hwDm3cpI1SA2NIyR9aTozXPsv2um7l1eCVacGhWcPfvw6sjILA+rZ/
VhWLFaKN7L+IAsEXmZAJ4p73EF5+xsAXgkvwoG5xWfQFHhgXtg8809hYGHj5a8zZ7uq9u5lP02rQ
cC6ksdn0WXACgppdxuG58+shgnXKzfQLfd06K7Sj0on6tsEwnAndMOGt4rzTrae03pTjjPGAK5+h
OzjqTAsYv8uNxsKvcQmpu1Ddw6SV94MHExTvhPVclWby5YjLycCIJQD1bnLRlKciDUtnAZlzKaj1
d+wZEyhlO3+6WYO8ZivOBxLZFWr8+nAZP/qBobvz0kE62G3+n0eg9r3AS1iqr/p/ovTttJkcAhYf
Wyl2o5yqkifXQKP89mOjICkGQT/LT0jd8z2E/+IKGXR0wYj76Z1VQ8NnkbjCEPI2Yh3xt4Xsyth5
ffxygn4tW6b6/PKYq1W/SYZSD2uhhW6fBfLu/ACUa4SUBrVKQdrsdDQ+H806o83oZkNA9sQd6g/u
AD8Jrlf3UeSBtaTlk8/mr7SQkOmsbZK6OsM9qjZbUNso3SmCqmfOmodhPpBM69w585xyMkUnqsu/
rE4UD8t4pzEx9x9ICaVVtHjfIKJUg1Utm7max4BJyqmhF7yYnL5VFSJjmcV3ZUH0RsW/WvdgHjh+
VQ3y+fK5F3xUunm0BmAOEqD+X4GC5NCH/s8FBfg2umJf6WD8g/PyuqFKpigP125ta8PFJ7tcE6VF
JZ3JEETTDuRp1ZZRBHVWTDwc2r7DgzWIE85/SWGeQzRd/YxoK1UkGM4bEo8vUXWvFVrUXklBdlrK
xRo7x4AEcm/4WJaU7iTyTvRweuglYPxIAF+TyDaWeDGjYKlmn2s2R7zVAX0xFXExF+FE8pjPWeuu
apv/DH6ZAOrTN7wps/3t8Vc25A0F5fK/xYyKvEFOgEx/0m2e2D25NgDdygEXL5kXOB1D9rlmD/0I
dd2vmunXXbAvzIhzJxDgVng31On/xQH174HQ1OxgIpqDq1qOlQyIXgwkfI0DxGzIElj2auvr4Go2
et+fram2KTly8UjJmCZrvQPfsm11EOSE18n8Vnq+wuHRQ3wp/gCpZ+RX7v/g/Uad65NLUeDQKz6d
D6mQTyGgneY6A1srmJU4biD/iN/aODnF3BbhH1SS45Kl8SjWCRge6LiAhDwaUWra8LIVCwB7Ehhm
0wqdtxTmoBvPdlPdkCwumCoQ/gTAoflQJq4ZI81sec7SPYKtK6xydlUSQKFIYNZgDLeZWxL0fEj1
DPGi93xpwo8+K8UVXj1YNv8XruHsm+sxtv82VCL4yb/wOCtjN/Fo5FQpA/nMQV4ELkP/J6eXSCoe
7z0ZJaAgLwHFO27a6oD9yqNaAh6/pFBbH5wDu2KKVglRcEaYBY5FdV/lFvSpi4/XCR2ElP1vWdoW
l3AAF1DuwckkmcW1ouUWwKHI3UP9jzv9raIThDCYadwPHaPNvRMRkYVrp48XEDn88O0i9cv+iza6
dGyn1yh6gSUZ46OFwDWNRgpGKGS3AfdMiVKfTvz+Dwig8hQ5DwfvgcsltOMGTsi12sIA4PkwnTNR
fD/v71DIZMYnxHLVD7cKR550umRlORnRpP9cyAy0RpYTEeQhqM6G7AQgBWjdn5N1NP1wfe0fgO7T
6Zei8hXbZFV1iZysaKtCA9uvmT4vrk0MB/Cy0mxdPxj4lWrgmZWh79nKfnoAp5/gHNL0X9gg0MN1
rQ9TT8GI/j6tJ/wlyabH1zCtfyWDVfV/yHb4oA6tqRjUHmTkuZ/HbyXgbEERO7J3Xb63IRbpzvc7
rZhgyNQYXyipOtPzO2AbY+ei85RPxpxuVuk0MmUoLorWvb7/CWbvP5oGrS4Vyo1Ns6t9B2L6piVA
1hUAgzVeayz1mkS57X5qmVtxnfGvXxnWP2+E/bNybh34t2cpFdYPLmTw70Jqx2S8XBXAzQ2XL1zF
w5rQVtLjw1O2MIN0ReO/LfqJD2boKx7dDGm3WDVs1CnobhWRt+zslVc//GwJPYV6D4CCEqegLmo6
raVTTcftG70GCa/Ran+7jOve7xvLan4nc/djBA2zgA83Ii2H2TnHR5T0TpC5Ez6BCT4sLFRRRlX6
KXB1XeqbSaLes1JLUxYvD8bFJnxf7DJUR7Pd6WT51qD2O9v1qElCgoYLc0JOSKL4MTlvmmGXgDLf
kHE/+A4vARTnajdaFgBW0vkMHuaGAX/4PFpgxuKKQrMKgrtgGZf2wXosmChacNopbtKBAAAARVhJ
RroAAABFeGlmAABJSSoACAAAAAYAEgEDAAEAAAABAAAAGgEFAAEAAABWAAAAGwEFAAEAAABeAAAA
KAEDAAEAAAACAAAAEwIDAAEAAAABAAAAaYcEAAEAAABmAAAAAAAAAEgAAAABAAAASAAAAAEAAAAG
AACQBwAEAAAAMDIxMAGRBwAEAAAAAQIDAACgBwAEAAAAMDEwMAGgAwABAAAA//8AAAKgBAABAAAA
+gAAAAOgBAABAAAAegAAAAAAAAA=

------MultipartBoundary--eFfsVS1Hba2ZnY6Mm8aqBwcs4D5PLyLFassH9VwgLy----
Content-Type: image/webp
Content-Transfer-Encoding: base64
Content-Location: https://miro.medium.com/v2/resize:fit:1400/format:webp/1*gHu4R7RdXQ97lahK_cy98w.png

...
SgIAAAAAAAA=

------MultipartBoundary--eFfsVS1Hba2ZnY6Mm8aqBwcs4D5PLyLFassH9VwgLy----
Content-Type: image/webp
Content-Transfer-Encoding: base64
Content-Location: https://miro.medium.com/v2/resize:fit:1100/format:webp/1*uaboiGoIqPNz040JGhbOWg.png

UklGRio4AABXRUJQVlA4WAoAAAAIAAAANwIAQQEAVlA4IEo3AABw4gCdASo4AkIBPnU2lkkkoqIh
I1O7SJAOiWlu/AkY/etQyf1h/pP4ze97xq/H/kV57/jHzX9w/v/7Sf3z/5f6b5G8j/Y9/W+iX8z+
2f3X+5ftv/cvm3/Xf6z8ivRn5I/1/qC/j/8u/uH9Z/bD+3fI/3G7sLV/97/t/UO9XPnn+f/tn+M/
6f+R+Qv6X/R/4r1Q+rf+j/wH5D/YB/Kv5//m/7p+7/+O///2D/yfFk+4f739ovgB/m39j/5/+O/1
X7g/IN/wf6b8t/dz+g/5P/s/53/XftP9hX8u/rf/B/wn+m/ZH//+K30nv3eKir6jZRsKETefQh0R
qoj5CIaH/pqNKBGiZsgjRNOhiAxAYgMK/EGCkpt7H/TGAsEcJAlFRrUReqXoxFk+YSJXL07xJF3Z
7P4rNVfUyzLp7MyTU2Stmo8S3h7SGTxZlw8E/1G/dmGXxMuCLYbnF4ehmDjtqtz5B04dKr/tLXXu
vM13VWh/6lQ/SrgY3+MNSxcrWzj+xnfmkcSxe2R4SYFxFvGYxT5yApx4Wh5kJIbYK5TCpgHHT0W3
5I7X+dFgu+VlIlHpZsJGJdxcoGIDEBh/UmA3/eqIf1eWY/hBVR01/ybDYTS3FgZpv6RyiCu8hmh5
pWIYxoPTLl63ivuHpp0MQGIDEBiAxAVM5NIPqTu5+qKfRJXj4FbP1zO9UmOaqY3ZmlRFDfW6y+5O
zBLeuuOQKhfpywRHAyamvTUD/BRDRdI5LuqHJ0bczbaSa3SsDvqabY4sxFjNJtUMPRTHfu9MGaKM
pPNbIFhYyLQyTAK4fkd5Eevb+lDGUVJGMkLki+TKr61gAKOTSi/51xNjFtR8ga0dTDIMDUdTP0UC
TXe2uXYwkqzm6XbGh/q7qJnWliZLqy97ADNPpiT0pSNhvxSpGZPQ8nypilNM8ri6BQYYoCl839F+
4Oeumc0D02uT5dyiq2udFM8OG+OBThTyAqhF+trNJEObHKLrc9dinyLgxkT9mDsm+PH14AYsyHVc
TrrTcWU71zG3V7s2yqgOYzFFGVyKp+qyz8a1Rvv7wIaOCO9IVwyLQyeGggKe/0I7pZwTsYVSmuZn
L9ZKfFHMKphQ3SGgTTTe9TxMXMehUnQ247x/OtLCFe4iw0kevFaB+qd8/2SooaBGhHAgBo60aceI
22ki5A8vZkUxKJ/v8Y6Nyv9+v+z8ATWH3yWOfRoLIwDzzcOhN6KCxxsKpQ3O2Ue7bOOlopABPR+K
gfu6ORDmt7+ESuKA8sGADoJjjuk7zW3DWwcdaRE3ZxxpVPbhSQEyyfoJXiPahadhd7EIQR7EcFaB
NHsZpl/m4bXCToWOpFWNhn8htyJltgx0lzjJ7iWZ2WnfIe52NhHOtHK6Yuue1B0orw6Ti3UCYWGM
HO4GS3XfcdtuRjg07j2ktWANUKTmXbeBYezpnhZr7mJceEzCUxcPOxsLCDVdRpSeifFgnPeXzqeG
sjP4yhEdzZEfMCKlVqtKJ6H7FU4R9wczGsUf0YyAz1WuHs2VtTot/sfqirEMAAlYnMD//YjE5Fhd
Gsuw7BwgB6OPiLYi/tYE0p/UOwlI+Nmv9eomUX6m+5dwJ+Lyt/uvyNr5AH7jSP/A3Q3Zg+VOFVKo
nokVwj5OT0xNOIIDNqF8NVP0slAcPu4/Cy2cUuaUas5vEgMpXaTTRKY68KNjT5LWoS3yRO0w9INY
M2yE25qbwZABNqSMsWLP6xge37Wt6P4lDXEidhdRbhVtzVF2yK+GWvUbeww7s1VpHtBfeis2e97x
emDareZwLiXzGorCjEZMnbdGufxIamr0LoYVSbeIE+ZX+udZd5HzYN3zWHIXlvcgp0o7f6MqN3wg
aCyffmWTIZKwxhNCtSHdLMuQ+G+MB9CK2d0grQJ3dHrUw4nn0tHxCs9ahYYb+kYVje9OZVrt1aRl
dT7WQFeu+l+YfpyIyUihdVjis0SkSVsl6diz/JG6o7L42Nqc78BicXuu/ftL0kGBU0GbkD7Q3Es5
cgXxjp1EGo3sloywzGQ/dlbAekT58+mf07OhuHAbNbGi8ErUsZlOGwq85k4mLx9zCj76jP4nYvB8
ZC3UN0FORAV+38L0HO92Wh6dhHnG8rEhLdNlKG8c3dSR76wu+9HugEfjbFm1sXdC6A61uh2dzMyZ
Bs6kvrOx4VzYwCd+UcTvSgBftLU+vlg40bj26FtvfOcRoOQ0A1jkkKFp3cRzOk1GfzXA1lFTwTXF
Bh540yhOXVOyn4D+qxa+SpME02OUeFbKmXjWqZvCfOTh5soRB8D18rgNlBQmAqLS5ZBORuReUMSP
MQGIDEBiAxAYicUDEBiAxAYgTnpiAxAYgMQGIDEBiAxAYgMQGIDEBiAxAYgMQGIDEBiAxAYgMQGI
DEBiAxAYgMQGIDEBiAxAYgMQGIDEBiAxAYgMQGIDEBhEAAD+/1/BxzECsJf+CMOtkoFdcKvF4fwU
Du2IHInN7UevzhZVgt+ml+5p3+2wKWNuZz3XJ+nJFPGpsQxaAep3vq6DUq1FW1/dfbDjeM5TYwNX
WRaFWOfPX+jq1ddHm0gQzkMD3MagFRbJpThFFtdsg/3RLY6LVSBYJ/v+tbgZiktcOwhyHeyNfLEO
uY1D9gCCc2bb1N8WqTZocVuxlPTsMb9SPnPPMM8PVl1eIQjG6/WQtHh4hTZ6LfFNlcuynJcNsMzO
vrdKXs/3QXIAl4waQQmqer9u5ygUN0JM5HvKZAwusEvXxwkD90WdgzXYSSFBgCH/xA3x3x1G0qsD
UmUEpGQLq1Cw9j1ukrCJmW3wTCjUbazY0Ix213L68oyGQgLUohcym0tl7B7rNTNvF+tUN9KtfK27
dTlYYepv63fMALxGkTV7WbfpKWuTkZ7+NJFe/QEvv8V62o1+QT0q332TMPMU35//6jeQZcNPFw12
FBcTyFzZDeE4v0L9nYpTegmA8agwBLkPKTTJRAX8ff5UGRPHOtqIoQeJPB18J2euroN33qn0LY6l
tVXjvn/JRPSH6ktZrz+rInDmYPqXbpbCA4I3n0fCCL4Ahlnh4AmEHsvNt8nwKzf2NqD04sCmFz0l
+MFp4xiR64SyLRIXRdweqpUnro1I3cLVmCX7wWfI9TJ3uaQuXXeyPIMjGx9hzBNr84otzahZd1tI
+AkiC6ylq3oGYTtptGg6cG5rPoCCWmy9Hk4X5E8c9PoxjCQluOZRv/JURVVqkQ9hadIo+hYkxKdE
+plyXBxYt3PAedLmkWm10wEXumuCCXTTfPY4e0UaKIwDjmgaRBcMx6anoDcRxCust8L9QhnyiiXT
4R0IhENyhPHOvSxXZN+jMPjPmyORzLV0uMTx1KQy0y8L7vSGYT4FvhoaRiPUvIPMx9s4QLGUB15+
YMvIDQ77Y3I5RuAufP2kUJ3kavrBd4+MzMYUXH//APVr76NwkKQky+GZ3PQY8y2UQXnlRi9iVvya
qWBDi+SnQMiFzSsVZthJ36dXPuE/RzNHGLg+15+LGzspVQ6NT1RDSHop7OUYZFoYdHxuP3o2yLe1
VJQfsPm+BvBNr04x9clkZrfoTcley3vFCixZLBY1RP5ovqwv7GICZcJZub/wrNqs97rekFPnJAn3
C0H8dCduUv8ZXF443s3f+yxgxyTLE2DVjX+hbr53vnfHb6zI4+dQoS7OgyedeVIDxvpWVOd1N5x7
1RLLVF1gaz5JiFSg65vHap2pRyuNnX0gPRr7VdmtPHW+vKzjunN8bjovLDaO1oS/pA5BcGGRMoab
LCslVCQf0nd1NfAu83Ffo2St3PXqxWWOy1djThmxiczY2qqgpR1a3zeOg1xPSXmHIeQDZ/N5wAYF
GkIsAPUEP3o6eChsv9mN9fN7NMhFzTM0+YW1pJFPVhh7qeSkoSo+m1RE607JsF6dRf6vsHKmCqze
RGK0kYpvlhLU9vh5WuFhsYoZuBXzU/R3MziClw1z+7LE5gWQjd4o9dmknovsX7yfmvnNw0qZ+DR9
3NAANQ7YIrKW7DEz1O0wh6v2jDXFLOlHQ89tqL4GaTJdLDwSUxfI4ASewf7mo1XZk++iGUV6v2w5
KqZ/EvMUh6pYVikK+sHfk9Uu0QPtv+XnuOgHi7HILH4jxRjFh2TgIUaXuwHfWcNIMiMiQMai/AG+
1b2xd3ZpzBVvUgLiLvnH6r6kF+K9ip6dpDEjn1uHkGUC4PLc1+nofr330adVhZOwWgcjm3scBWpV
xI+RdDxLYjzOneVSmMzYPJlJ2hHmxtqkjLgW/5e88whk2aqI6hxCKV5f+HYxcrOOKJuo8RyrIGie
CecB93KQOVxL2sPYs+1PiuHrbEzjPCw3KnEWVPdzDZ+Kbe7s3oiuZ11okJj8fbc24GoSXvyScQkc
x/L0vcySpzB20EV4edY4sJe1P5aRBH1X2ALH4c1AACLPD/hfQKw6eMVRHFwup7I9TJZlwdPousoP
UV+RirEG2pIdwf6iR9fVl9o2jsUvprPAhzu4PIyIKTGjP/Yxg5MFEtCsN2P3Ln/Qmv8zWw0MU9Ai
M+/1Z3n8poVthvgjlhyr3Xni0t1PUN6UGS9p2Z8kdgkql0ePD+Biwy+uDjNKVxFxjeu+YsDvQgrY
uT/JcXqtoPm6U0aIZOz6g63t/BUSbGU1Cb8muQVV1WPXXbyhNj0X5c3gIYgoVTxKlf9CcYZsk666
3l1Wl5yGiR5uUpqwYfE3otzW0nbGWV3Oj7nEaVdqUNPH3ecid5AMDkRstD2M/yAJ113hCLMNFzSh
l0v1W1h/hcHterPdmIokCwS7Aes8WMaMq2re+XeNTjBftxqESsgODNLwCuZijbvd3zXtWTJjNIns
aOjq7Q3+93CPx4G4F38Ky/r0U//gucbm3jwsdg7wxcFNarpGL7bpj7bvHCit8blbRxNvhZGuIVI3
jDztGK7oLYnExcXRUp8mVPE5jPgfNpvshDGzYTbcnS10BEASnwHST3W/CEihcEVsRIikOJXmZDws
fwKDwmG5mWMnjf5Zqr0aNxKRiTaPWOzd662QPsuu7ab39fwulJ6iOcLAsgRe5YZdSw1BFZHkUVO0
n5w/PJXufDnHyXv9RzBGQCG5lYsFXaORaoEHjr5sPHEzOsADDYaEKOmOrtfbJwoM8pshukifIfyQ
+P/bfdpCxnS22tAUCfhV917nF1kDlAAABxLNYhWJ/G46hv0KbCaih9cQloL0IUKeaNWU1s0AD4Vb
hCJWpj7I0VMhdsbRME53GeqbE9Ea1Le1FaexP/E/BqWlF3WXV6EV+FcCPO7PN+Iy88ZBPd2Vy8Xo
JEnjQ1FknoNQMKu8m3m0UyN/ot2fCrMJybRBHS+emjoB41bxAo+bhVts8W+gwT69jJD0/q+WjiQN
mZfZgnOk5kDCA649RhG3yt4hzh5AmyP3QE7enb8pjwd8c0WwO+ehVpwjKAMEBxAJLEQkSRCRBBE2
ikq4f6kpLYd/BqjUfUT/9i6jL3lvuopP4m/fLvCmJ+Vh/aKMCV1Ga6wSAO2d0cAj6EvXm8EUfHWv
/Dci/BvVEf3R5Sja6HCC2yU/+FcnKrfw8kzdbZOh6XUn3E066HoLnTLLxy5uh4HICLWIUvgyinxT
6VE72+ONXHNO6/sMCxRTcqJAVXK7v8j8GbQqHRGyiPG6NU1fzAu5+zHw7vihOcX4pQ6zYCtp/RG2
UZyMxExD2YV13K9DNc4ZiXj3dWdaN2S6Zme58IqaU1SYf5giSFXxrE9llzaZdq3kd95cqPYLURkh
GU4Pk0sTdSnvz2X8DbvIHtIQSedV27zcJy6sEZVbztLBu1OSlr1p9szc75cEMaZGPskKbScGLBzp
2YjOeNgzNEP+S1gncxYpzYlrKTurJM6c+coTploBEYGy81dJdpI1fHhG5TgrqKqHyLrRjbVS4Ru8
AYl+gBUooEhvU9//sFVksmvl0gM0qHISHDyqm8oSF7hL9oDC1Xx0DID3CjRhrKdNCWVmytp4k4ha
piV6NMSdKp28QTexK/GJl+j1b1woo9s/bh4iV/VT2zqcLOewLGtQiubbo2v95CDTv9/stgJ+uT2b
Atbr7GtsEGjBMNlkUfh8g9ahhv2t9s2EbqNaSt+BvQWaGvibH1RRzeJI/zZ1KOzjndXnL4XUg6BA
hgebA/Pnsec6ylipvLB4Cnf+IDhb4DPyw3h9UeVop1c2+nEMuFmiutIiFuFgMh+BcSxOvIjT0Hp/
q3SaKG7Vth6TD8YLeZyI3BM+s2wpcoyH/fmMTL8s4jK+2eIKFjJLMC5YSItQM8E8NDAg3puiS28X
kK+rFkFYF/xZGdIzCiYxgPAU82EvlG3vOBmHmY1i8onAcZdbNf/aaDcGdDK8P6AVwH5ypYmdZl8+
R0M0grhYznG4Cdu/5iokuWKpech0JXh/DQ0o4HSkt6mrkKtE024L4e2xxBfoVHpLgW2YCTiNUM1I
hHZcemnSlkPQ4H7i6TT/GxkSG4HUzxYfGBdEkk3DM1oLFidSuYgS+12+Q8gzjvpDDnaL80bmiIxQ
kZB9FtOooP7b2fVbGQzjCb13ehRZva6xSacwdBlPnJpKk6mBny4f79P58hqmOY15rGUPrJRt8f91
FzH8DgucWdYX0kKThoJZfeeAxwtCMJy4gmoVIfeWEHsPHfbUA5hFywBeJQ0fH0I3tFNUsb2nZxK9
W+gcOzULS99xcBs3zEz1jsh4itygEFruRmbkhWZ0E8B6pOHpXThCwAFfmpUK0SLsn+fzwul5Z4/5
LrNLkn8A7ob0ifgC7MI11rXvEqMWxE2gNJPmH71+/iM/ASj7IDA9dB02aUlDAAO5fw5v3rH9jNCD
GDIYrYi8lG7SxMCS9T6O6zi3vNgDs8RnAOEyABAINm69t3kmlJ/lTApifNpTd4vo38MDkVpH8W3p
sS20l7nqyt1kkpFd/V/4X4yhJnCRerztPrkgUphCnPbN4+QLqD/OD8LsjtWSHbOWc8oiUA/ROBdb
d0ZpjIcDqAYJrIU2QLxoKaxtQSnMNP5WWf2pBrlZK41Fnyd6g1pPtFzyKRJQt9F7QpW66EjioD+Q
x2hRU7WIePGqexR44L3mp7g2LYYr0EPPPqUyixWzH9k4B6moCkHP3oT+fBDwxBGpwVGxt5fVd10A
teEEuK8feCtKFxoyAVJcqvQlcKZOdbtNs6jJgzc9b0Oin2hGYI6/8YkK4PeOABu0/IrTRbqkGY0y
w4OkLwVhebNiED+om5/qXH2fFlqqPMFhTDj6UD4Pwl4wtAgbgHAck/8fsVlyd2Z+dzzI30akEXHN
HYKmJkdbKzm4pgdDXbuL8t70PCINUqgCA4YUpwxpg8Z3byaCcfRuMam3xFLztPRQYOb9Nt0ZXBT/
j8FaVLQMIlfkycK9Mj5G2k5DnXuAPmEhTKbW58aHKSzMLAESAbq2zuYpoOftHNFP4fLCncE7jANc
oWPo+TvTpCupeW17WE4AtIQFRb7X3WZOdttfbsnewNQ6+R2wLrCNGPetU6rN2MEAVudCmpc8Pm6X
/lGvoEJR3O3jlxme6ZCBhg7bzbH4yEWtqMo/PTf7exGkRjmwZL1iRHQUPuuRMFupa+rtJKocM63b
uQ6DH0ERCFkkccGvKizJ4QiBNbp4ytvNtRHzeeAKX2I2lle+KoEsUcuni+rGYDNdixWcBxemg+G9
ZKxwv8jUiHKSsG23rUnwLgGMA8bV4Djg9mHc69tnYy5IH/O95BK6hshixODmH+dTrs43IkqYO4zi
lNMG45fvzZm1B5Eafnt2/5QgXKn+uOHLEz3tvIjnHCNU0S11lokEODMBT3HMFiGUbnwiWn5SI/BE
XIQ2ZUmpufGFv6aFu0qalUjr2zxQ8hQzeTWNjrC3bTJSdR2t40Jek3GoI8HnmRR2VAfPQo4ovRhD
94Oc8LsDvSSwhcx/6MFyv+ZIzPROl033iwNXOdXNtbdb/q99ONpbDpYQtdh8G/p60UmLHZ6m9ElU
TKfCY5JjYjuhoOfSs176bm9kjhEt7+n+6FPxhIMv1kqmaAyXU/5K/x7C5JHeR7nz7VgNIvuPXLWv
iLrEQx4JjuXm0cpD/kjOw0l2M9dUnWYHOZT909qO+0i9Xljm1k8F/BkSCylaImQt0r6pNIx2SqIm
zhMaEvo++rBiUoiyGTmGjPw8BsyAIELtpREI7KAiYG0yL/AlwJ3ckNbDpY6/WqW7mnZC244OzvlZ
eniaXzsy4u2Fs+nRdUULl3tHj4I/Glbx+Mh46/LzX8fqywVrtPRMr2ipH+Qs9JcDvw7jethNzZSC
E1haqBvLngVGkA5+RMAB6D+W+7pc2yl6xXBC2gq7vQzNLTjdjsJOw1AdJ1EjH/m0rkJzr1DAh8nh
AcqXJDYUTsdAyqCNVGf0ASrjylFmJCIj8M7nY67Lp0Cvb6t8+9JB+5mi8vX1G2T3JKIhnYXOA+5q
dKrB9oehWwD4o9wsRCv99pWuwPNXBeZijfzksNQucufI+4B6Fi83N9vKjQsyteQ4dPs+XrIyxMZl
aoMXjBL/7awlF9LfZKWFaujwKDwHYRSOnQcP21ZvTveJAPHcPMX+RN/E14hq6qLlFVUV8cfgl5+r
YDzXAR/59irVguCvy9YtyogydP2AOCtDpUfAFY8S7Q6vWuxF6h9JeVjmg4Kqy5o/Sol9zzeLC2iH
MGQg6jBrqagZz4A/Xi6mhW8cCErbcPgtfiz23TGGjoyh+lseIAwiWgART3+YfqeRAHTYrRVHcfER
V58P9g9sP47Sw6MlHV29vi9PZDxBR1s/1Ebm6g4OIUPJtcF7plnk+iNVa9IQHOBduQDyOFxlb1rE
RH6mZSkE7RrcefKP42h14p0a+VQPa4Z/r81pa9b1bNFiDsIG7fbeOVMY+3K2TVYEueY66V8UCeAf
NNNc82qII9Y2KwcQJMZbvSjQfhyYuIniKVGD4/9u6d6fFjem7K4y9ATUejh1faYEcR6YaWNvQtJ0
Ydzk2s5UKJs8lMVh8jFA7xDANEVsXDeUe2O+Egb9oN+C7MHsjivc7wJ523gejTrs4whEaiGYzVIS
9YgKd5iHMZP0zyTfA76wqY9o7iOoAscBU9Nbo9h/Ud7vSBcRT403pk+xpPmx+Lt30dpKz0iq0Cck
P0IHj1X9VvSMSoDBR8VqOV62HLblzHwGwANcGlsxBpBxMMyp06DhM9NA2gIOFWmvStmQAy8/5Qqz
q35w+7Y52I1j7wMuan0wkI1S5mehiI0V9v+XtlVyUWukexUMnbwqUftiVu5drmYWn/86i+AdfVs+
3pUF52kYcxNYYIBNCKvA6kCP/0L4dTEvnQw5yiUbW9Ier/LMjYVWGabN8KsyQkD8q05MP0rKcIci
zMFvQLTvjYKQmLzcZkHBbGWc1/2ABsQXKky61w8hz81Vp9Hg9CVMs8JD2MnkLQbESnArsZu036YE
uqzgF5uj9QYhdOH2/+p/ipFJsyOIlnYNVFp7K4TDLoaCt4Xd6qK9cdScUvtFY8c0++4txwE/NmdJ
MS21On8WUtoEBPIfk2ehDjJOxcyddcMaFsMPWOAdrThK1RmDjR3i8cemUXy+vSuxsSPu6H57PEHQ
AqjDgzVwXnRSLIAoTLw84bSz2xrm6cFKptzq8LlzTpbPgbizaOEUevEU7Xfog8D9CzLJzfAa1z7c
VSfj2Q6HnsQoQB8mBHVv5oj0oBQwYVrwx24s974WyCKfbWhUD7qO94nQezMC5FXk/BHDWJ1HVOm+
QVhZGQkHDeqDJe7tdjJXqrt62OZJSYRatPWUIZHLb0UDMqq/XmG7xNzj0N/C90Tok8ZK2/SGehT9
ocnDbDjVpuaLK1etyrkEZHFyDZ3ta2SEwRMAew2lGLvq+Prb3gI8JMeqwAKgZBgPFrKPE0iYmmB3
ZPuczRznM2Z15HS2XeMetjhdwkuBEBm7HqG5MthjStLpZbrcmtic0Chp1nlpdszcV5KZDpUhymjE
/NMCSj5pxk3uEOC4nbarSkJHWeSA6cxDOWF7uQwPQI/YngxliKZbG0ctZlbfbQhgk+J598iqAZxS
kaUCrpZb5hfH8EVu6+s/MEYsIHgdyBnD/irOWgh9OFwxi1tRS36yW6JqGDk5ft1QHf4CAlUcKF2r
OsB+jJsJmD568vUcLP4eIe9Zg23PHsYhKIWfczorqL28iCQUixr3X32eOFWXU8fEP+EFP5piSrKl
utOodz4ldhPlzwD3bYKvfolZypjM/z1lMeCEzlBg7HQLLPl/tgpZ0MaIpDFl5tlIOx6UWfGAxXQw
gTpSUShkSG9LsYyW0aZBnBksZWxFfCQFFG1mopw30gtRi61nxKu+R0WLmkyoV/LND9Z3jqEfbMrw
3sFi4sGNp+jF68e7t8f0YkKhglKVV9sOmyG93jDglyJKMczT4nfpcn3FeUWz5uhwy0fD5KG9zp94
CCCALyVA/wAcerarHX5VMW4S1Tx91mB+2LCXNg2jogQXbWZOKijwSa55gksQFq4txSYllup9le1N
vQo3WHpOc/w96ifZ1e5fUCtnBE2cTsNf51rG9ZugUMR7QYit8OG9wbOn/7BKHDDzJb1EKbxNOIPA
iXCv7s1WDe5be4OT0Z6eVyfNvee1q++AgtloIeAj1pprFJCXTp4gbd5fQoZTww0NnOWBiE1QGOIC
YLHFiV20CIYvbEJ16SOcQoI8M9P7C8HIKlXaKs5cplIGqOIh+YR/D9NgHOUXucs385+bYC0VSr+k
mZfbkx1KQIExiqfMfaDovEUaL67oIi6coDbHt4r5ssj4Qsf6lhF0kSUpiTmbUNxZbjl8Rz5uPohG
QDqsa2zWKBbsMMlAjRDRopezKOX8AfE6WbT90zxaFFS2Ev3Q0qq7j8m+P9BTceMQt0vMk8YImC9R
neNz6FljVymYRjnhc8m+ysjIYUiAKFa4BBLwWkGRcpHD/CIKy4L4Y4Di2QGN+Ul5Kaey/ryTpv/n
/w1W715eVS+Ltp+pcKCTJJGsIDhdEk2uQuUjKwRMWTgE5YKiwFdcZQUa+RwPf5DWy0Jn9Y2jUCY4
pINWrQE2263ey3/Bj6GPjC0p0zhhtGI6vzcIc7ipMeTOQ4+QQMEkLVbppgS0Ne+AlbUzQUjgnEXO
ZEqDs70ibP/hVYUPTT8iwS8LPEIu5YnPvbSR4RrN/ZpIjZyWWCR0WFGQ2NPQYSgvbM3pP8XUG9IU
P2VeYvbNFmFwh36xz7QMnDuaWnoGFjpQbJD4XqOpYhS+hNiOOh8/NyI+U6/+Mknd95YHV6YyyZgd
kjLqliVAaOtEt4Nfsa1BuLPT7K5016iPmN2AOCF99aIMKHgVj5O8hZi5x6OHNCtGBfABdTh77mJr
HCnZmdMy2nR3WOb0e+LlSXEzs6UC8OxaS+czt2aXyBk+phkp75izKBwQCHnUy6XXqgrdED/bJ9tc
vGzpgbEk9MS2rmR9uGS9UD1C+KtY64B1l8c+4ig40WwWAXTgfB0AjWET0LuJOlkwaOqSDL2D7UhC
0ZaeaGnglY0jy+kJo263eAOA47uGegNCdelGsO5iU5LLjgxmbU6OGeNgeoRYLGuUok/GO9eJKbQN
KNyl0DZlIfwsT1CazPBCD1lMp4oMDdnGDtkDsT7ijjtC1buevLRTd5hmjBCbHCGjgzE1S6b6d2FU
UUGFO3o6Wmt4KHPfQ+eZAFM8Up0G3494nzMjSTQyzQKiHWwn+ovT+MyqCRhn3JySN3qW2iGJa1by
XUQyYuGxTMWDPVwMH9dkJHiFeunSe857ycpr0nvZmS54unYNv7vddk8ezA6xrzuwBJvHpZqlyBTC
ybMLfcO4L6Uxs323Pjx5mHr4EQO6iNOlM2s4cUr1OCj8VEVm9GdDcwG0s+kYfBBDtEguGGTXbKdE
KS9oQ2ftt5bF8GZ0ZySDwNb2GQuqURIz+NGyJG4y5HlKLEFRplXroeJUrug2x1AXhc+SG1Te/7yk
tonOv7Jt+WswgdChGxoYbd6E7iR4YEYVGLedkZL7UhkRSZ+Q52bywGcHICQ8P96jFjUCXSX6ti/g
UTmyIdIUrARsl7LDx2+T2iaikNed9ksOCa36SbHWwW/hDiRYI6VPZHzm6HykimiFonh9cStocOHQ
IAtdecPTbBB9ZkdgJ6CmFPKUTPK1cd7Un4c/p6qHXc/dVGl0XwBgp4IdET09XgwHRLUwSAbPycCC
dECT+O5BlHzC6PRkVwLH3+KcGvSju0V1MYEP2AajGtVeWXCPR2K6ChAIXaTi2mWqi3MqDTySU1Zy
1Vg5Nrz5/5JUVXvTsuEeDPt8NjM10cc76V0rJ5OLAPMcflC24AyB+e4BkqyrBdDdY5XdPd/ID54y
X8bkomz/w9lTrLownF7n7Pc+XriGY8bQ0nL/JwCcljjs2N8Sx+XYxtXQXsHWT7VqVpBY4t8psnAB
NrueCScDe45tBN9ebMsCfB7UmvfeulxzyShZyiO+QFjXHFAQ4XWquWr7gDRkL9Vz9awlAdvR17ym
6Qi4ySkcVetcjDBXjdZGItVqF/wq5LBaBR93qr4Fqx2DENBMf8/GQvdUkIXDYhQVMWjw6/xYZgD0
nR3TXKHBVvKSjKkHvlYiIyxXkLikA233lha+6PeAhi1Cx/csg1VF2y06LCKaq6qR201LzD2is7nH
eNG9WhMF440ypbcmg0Yp9fMD4brpCWxysHcSo76X0MYc80GxxeGDNCPjg1TlbEzhA0uChzErp9n3
BZe3vur7v02oYCLi4flrB2z0dpKFDSTbQx4SLSHsl6jVykzuwsMQIJ3nCh09f8sL7tarNch2EmNt
R4aze16UOK+j3RdUXhIfX80gGZPeQab9PwHOx34gFjaQt8rHpRWO+WL6qv0/V4iMRBbRvmdJ2ty5
OwOSYgNDy4fXUamjXwwtebDTYvKkKXS/nd3JzbyUMRfSgPpGrUIosEZZxyeVU85OW2YHvuUEXvfP
yPEommNHbqY4OYvdaaY8A+SfhqwDZo7eewAJEP2/C9FFlXdvpXf0AobxDudxq7/DVcxkdvZashyF
K4iazCZI1gnDSYq5KODPoJa420g08Mj4tV4BSuUDZZRKAuBCoq/Ppl0wEoALJusZ77lOEjlkKphp
FCOG5kWFVumAQU91+ly0/1i+5C7uBjIVGntGlnKjfwcXDRTRTXFVSx/c8rK03b3ikcynz8/agXkx
OyOagnpIvs3KBrZkGj6Z0Yz23EywepEkpKKoBSccLVoiek46pou+M8tBApoHmp5lHVdcjWJ4F+7I
+U9bw0NUM9ja9UYkMUTwUVWtvuUviKQSfHl162RJ8AeaZxTaJ5v9HsCJnsTGK/SyF/KZMMn5ZS9T
M6yS8BTybDKaOdpBini1/vCTzlRMPEc+3f2DZxrdC3yXrVyUfrbOz/ILzwiwApku+ogsCwJtUMUH
133L8S4kEuoSIpdR7h49qzqloKzMGcUk3RLIyAOKQXRrWR4Nx3oVjZiLsqucwyaNxD4k7wkZfR+y
u4WP/dzS+okJhfue8jQXcTHXhyuS6GGxluusf4fpmzaaUef/L7ZZCO1sG+HWTBEAqJts8iSdMMXq
qKXg9WLthii4+56Cq+DmCr3Kekc+kclwDQLYUqydjehVdcyPOot9/0y/31pDiEE4kvuolz7+A+Fy
WNwKx/54M9l+nfqe/WUywl5W6IEkTZB3zE8NnJUw4A9BNLys2GFgxZbRXecog2UW1WhR/F2lb+8t
5CT4wy0d8HvpcIeFJM0wg6bA6BU47uV+dD+B78H7NmtuTwZeWU1cJXiDrw4F20G0LnHqlAI/z91A
WLw4spwKbz9kSb7SgG5S2QrMpXfrwWIC6lIM66iIjPMdBmcOBkBK+zADHmN/IlsYI1BIuZ0sqAKI
oHTOhbjVR/uFoF4DyK7QWtcoaMWKlc70WkumNcD3L/+HzP3tAgkbRJI3r70uvpPXdWpdIkJ+BlOt
bwHFUeMSrbRw3EXpdUbcGOka4Vv43hUALq2hjrPszl6JNv4U71j0UDVd9MwtW2aoDAAE5+bpeTuN
GPvm0/zPlO9KHDrRM5Fvj+vLfF/NqIxDTC+XmSkdEvCovgFBuWinVUrkyuE9pvtUrmA9nTxM0hc8
8mQ1HZuHYV+WltGF59VF9X3/Oe6vVDDAvV2YxGAPxKBbzUGBcEJZ85eENJFLbUHgTmyWu09OtBDP
hbu1egF/DD+YNLjdI4qsZdZXCYXQiPraHBuH+tsjLcADqkJzksSDBvdv55cm+n4VNR2t1RLndC/B
JBhLPibq2iKStdQpX0TJSUh/VwHPvfGaeJ2jIyJw8ZhoxuT8ZB3nA5MAoEnnhBJp/D8cLCB5egEx
DOk5o0F1dQCOqEOoDyZ6IGjndngcY9FeSHiLivwGYgKqeoYokAxYF2Fpo0QGDmozI0Aci6cL8VuB
oUjcfWMiJe77E04x9vW/1hMJt3MbqCaE2nPSl7A0DbBsnsnGESXAITay91FjOuu1JQqrrZqUC5Tv
mcCphixcn3J4lMz/A8QIccW4EPMKNP6DGdQmeTYKCD6Ta7ESYEnW2vbbS1kUPb3eKWBF96vnTJOh
a/wjb14ffu3iv0Z4KN1OFEJ3VSdJWOQbqwdmTCpUem2XVdz1eC8krlK7b2dOv7T2YOeI8PmnXkux
IqltYjyx/3aLXAkPvnoGZRQG9m1ujH0eJrQEkKp797VGNf9YDp9Qaca4XsezlPzWq6oMQgljXf05
dficlbDsJm+EYobSCks1cUhB9R2679KWJ1qvsTeIRWeTBrlZsrqFhuBb3k9mKzrRpr7yyZVKpmD0
efn428o7Shcqdr8e8BgLPssU5JVxswII0AdANU75qlO1dp3qHepzfFpd5m/SgCLGFJ82pJ1AqjlY
Nr7Z7Hd8ooIvw819FAcPvorlwxe46+AbDZvSOp047kEEB0H7seCT62vK7cucZj8G993V9e6hoD5/
giV5oLmphYOSmOqNR9g+YxN9cj40JiInJ4/SmzWrevyHMEJ0a2PpDbb3Ait7qIKAn1Az+2WZ8mfi
2FXhmZWL3WmSpOr2FZ0+J7iTq4+bhm+DYaLYKh6RFalV51vL+M9XvIPuQ0g7zPewK183JPwfIubJ
KDuePfN4G+zHYrS/fFjb20ZPjn6DzL20srRGsTnwPYnt9T1nbGyOrCXpFl6Lu0hNyfQRroZQuzdJ
qWvPpXmuPpBThqlsYV3WMnruLnHptrem7LhkssAc5BP2w9TacyWJy1o5wKLLmkeFxxE7ONiz9ylC
NhF0EXF6Ky+8h3HAXzo2bnMnGMcskhXf9GGJTMuHZiVA5oDWHE6j/x2EaViafcz5ssTs8HV1V8CZ
y8jNg2lKkhyBpIrshpaZoDSlopWBTG3vFBn0EPcvei6CKN9xxELI6zhWeEsZx6CwazWSdaVq+LqW
DOPLeYcnDDxpkadO3Qxb2FmPA0c7IY7leDVNfR4hITU4PrYLTFgFcrvKQjFb7HpGENi1b5ICLDw1
q8kF1pGomrg9UacPCjfZF1dSAElk1aX9EM1/ZQVIHEIf9FWE7do7ub6Pd3SSRBAc/DJ0rdjBD3d+
UxHfvng1ZN6p2VQxoNPRGg0eBRBHvMQvPvBuRkRbM98O/dqytOGJANBt5dMCX9DzuuIf2xpTEfQm
yyrCV6T/DNjUlyhyYsofVXMU8CZSRDy+x3haWZd9/nK3z6W6Hm1beKkrUTod3CdLjFK3DBvXmTmC
gShAvte5RZrcBPmHm3JmLd1/JxPqopN41Kst7xI62TJahfK0HVX7Vr4AeG4PQdf/wO96a78ZGUiN
K1fjZIAeRaXfJsukYRGhE99tzZqHBRiNgVZGNGfqeuNXYShf9dIUNBhPhNlcMr5Ijglf6llJAQvK
r+pA14WmXDOXXN7tW9Ln9MxxlI6HrO/xSeg/Ar/dAZxH8Y5/dfqUOqBJvNUJ09J3Qmq80iYvUWJ1
UEhajnjssvmbGJSVJgQ+zqbpbM+ccNIZ/HPZIHbMaGPp8IruiW24VZUdI0CDMmoscZjBuTRdMnHd
i5PZk5pRoNloiXwVvPB42mvaBgQPsYnIThxZG3zb90pMEi8S+neFKCzoBH4asLAPSMKaMzKfJ+JB
6W9NVG7wZxUnMmy09thq41/sCpin7mdCUcEyL6whuPEbnbf2/SglNB1hd+2dm0S4nxk1mvUwLq5x
yQfFLpkatKuqelpUmDUlNfdyUy024Qf129IvzMtcTY43BU89hXKqXUq5FwKaSxbCxJBKrcJdi906
aFoP9zI8ARpCDcV5712yETDnSjJ4E9gYvwwxUxnN2986AegGZzvuFR8JTe8Se1BpNS9BBmdvg0Hn
I6NNve2ugwDIeokINBvUHA1eBAw9lkWG3W5rLJqjbQHxBJwCg5ZaFtQQPJT7lWcU6yE0/jQW+Ccx
0r51de4H2OUWpnOib/iiRjGUVOjAp3xBPjQoA5yvnYqDjdpHnQOBSjTyeQ6vj0eDADDq5RVyqLfR
WZCUFskEbaN7QNMJ5FWn2JZNijRIMmWaH4Kqp3R211ZaH2oKkYX442pp+ID3Fwz4BH3povtJSTAX
svz6U89xlT1umlYnuJA4gSjPUfCVAWyg7TMGhFuO4l6GBSB2RmwrRFm91Pz+1tE1T7lOPWtH8F7X
T9dfRvD9xh2B2qARd5Z8SlefJPvg2dC7tZQo94VWoB9LHyOWVHyB0rlc9eK9RVs/qul7lk8pVObr
LqPXbKMahtJpWIO7/KYHCObriqzeZeV8cEKuLyHr585C7ew0V+G6S7XOT1HeY+dJm4Ymv4rN/OAt
xdg2HThgCgkaS6Ley+sSd9Pp9pQzglde+K/FT4JS+7MsY0EAiIe48loK1YHMk28juDwlpPmoNt6R
ysDhwtjGbkEvIvwiedl14yVbgv8gPYQyVFmcS36KRykoQ/MjRkeuvXmvgJvOA4AR9AwFWG5CaVye
cKuSFQ9N/2C1qX0E19ryph7I/GVe/mrJjuOPLzRvCAlJ3xAQJKxxkGCwWzNbAFwPfeL9LT4EzM4r
eWov2L1WumLdiOqd1rDUHesBXX9XeMbf3cfZSFDjWxIta3bMcQtBpZ0Jj4TX/z0HqWIeIBDyNk1O
7iUQpzIgtHBwQu30hNb2lRsvTafsFOnHRIlHwjXWOa76kHoyTmuv+4wtG4zDMoh5/tGdvJaE/5zK
xGgZwiqOLjjbyUTsZlwAeZabepZHtiBXzPfSWTcsBAo+SSRgPwKuYUOjRZ5KJZkExSFofKkoux3B
l6kBnvhrxunbn9GCTQ9VF9jtx63TSyJePyTUIM2dTwEYtwW9lXAHDO4e+1tB0dwTb+Ept5JFZhTd
DEWAma1hGsICTQVIQBLkfoDK1MpRTZBzxXDcnMRhHU6XgTQcM6+7nIxCjWW+waiZPL641xoMG11Y
COAPfwKIV/zHV4zwMAvddXdNtl30L+HRJiLNJsgAXSws1ByYXZhw25e+Ad+abMBqd3QYHGoDLrE0
BJVQHDhyse42qVxnOmnNBa7JYnfgxMe+EPLYu2DhAT0MpDpuBvre+/QV8aY7p2qF1viBRtjou5B4
0RhojRgfzZsOHFNElrsoaDKJlJ4cCKhveGBja4WwWPKgH+g0Wh46dzFaMkg7+EGFnMhycXJDvFtJ
itb+ymqwRKHQkygtv+lIPYZJMnZASVruTXY3r6+77FA4Pw/YIVx8zLfKnwlz9rZMUu6PBMd+WAYQ
9uzIu5n31NoY6Y4nn+YSbkkocMTusjGcD+3AvDwQ7IoFvC+d/MjIBv/GHERQPWDBLZQzEWGCYwX8
9sMHNoWRLAnpR4OYYFKaYTYpv32QSt1YlQhpwWqG3rlKwgOxWw5fAgErRArCWjZRz+0VVWEnWN0X
R5vg2k5cZ8SzN/kvz7mmdpWVVT94POF0UhJS+/1MsWOxt1ZXLoVpPOxr7Kv9HBIvV5mLsficXZtt
6DWk/HyD9DNdEf5bew+h2saJTdR4uMhJGCmd6SqSbZZTOKVwEkH5tUYXSTjJj95vsYUKrvpLTxwI
zdC9eiuPHqOyneWkQcAf5HChI6jl8akMc9fb0NwwTkMLIuJKcDxH5kkuSDxof0yTOr5Hd7ZVJ/pJ
i5XiP/aTFr7qqaakDxVqP6VVWRerMEGoFWk6oH48071/mJXLTA0+JI96QSxG/2ZTULeZnER2J0a1
zJHm1nLhaSaDsVhgyoiyN86Z1Qkch8qGiXcu07ZLsXwf9fBFxY4idoT6iUjASStLAuJRSJrqD7YN
LEI6qJhYHo7XJVVdhREHAKEw2e6sIWKU7R8VHMJUSrF/FcqenXO9s8swZOxxbjrCBEHuZ1LFatAm
MB51mds+yrTryYXwvx+by/2OP8rlF69CwhBSKSA8cSjFlxp0V6IW75umqTPNr6/klL2Pp23IWwF/
HYqppjkHSeZZAT8zhtZGEc6d4ofM5j+mtoeMUgc3KuU7c7kRZlvBANz7xC4M4ptO846CknxjYAd5
VIPro4YET+p2OgJymHFCM23m6OesnTDh5rCOodV+awJy5JRAgEJtiev7dk+br5sGEl4+aoodwboM
jhKgKAikJz3QvCAVEitZNWSPD7zIHc0kxHgwev+h6TcRKsv7gSr6b/r1/r+OsnQzqD+K4YOaQIHg
BKk5lIQdtPWaBsURIJWAmLJ5rjvbvei7COPHSB+EoFpO8wONoYumiyoBPj7HhNwvKmI2rRqxwpxZ
N42hPcjTj8fk2/nSd8U0yix9H15qrrGk6dltik50XasxzmWOyz9m0vNfJdU3rQhTZjBcK0HjoS1L
vM/lQXKaYaY6MgE1fAnib7g9WRjLR+uUB0XsJ3MAI61DBb43XKjd3g5bjB3cIsbxaIroo/o6mS2P
H7PSFdpotSwz0UV5o0+jrN2/bAKJQDbXBDpfnvuKgU/4PmNQL/BPihT4eYxhnT55F00VWqwhYpwQ
n7o+ZupDkL1631Z7fQbBrVParYqiz7jO8C+6aDaJ68xwZazSnGocEcmmUSEahE+AAAAAAAAAAABF
WElGugAAAEV4aWYAAElJKgAIAAAABgASAQMAAQAAAAEAAAAaAQUAAQAAAFYAAAAbAQUAAQAAAF4A
AAAoAQMAAQAAAAIAAAATAgMAAQAAAAEAAABphwQAAQAAAGYAAAAAAAAASAAAAAEAAABIAAAAAQAA
AAYAAJAHAAQAAAAwMjEwAZEHAAQAAAABAgMAAKAHAAQAAAAwMTAwAaADAAEAAAD//wAAAqAEAAEA
AAA4AgAAA6AEAAEAAABCAQAAAAAAAA==

------MultipartBoundary--eFfsVS1Hba2ZnY6Mm8aqBwcs4D5PLyLFassH9VwgLy----
Content-Type: image/webp
Content-Transfer-Encoding: base64
Content-Location: https://miro.medium.com/v2/resize:fit:1098/format:webp/1*xZdrvrVr9FxM1Q1Ov-8SyA.png

UklGRiIwAABXRUJQVlA4WAoAAAAIAAAAJAIAhwIAVlA4IEIvAACw+ACdASolAogCPnU6mEmkoyUi
IdSaAKAOiWdu3L25JBIA/v3rvAc9nyLDNnhajL+DM1DjeXsgbWiwftJOJ4/+rlhr945W/xnfr9G/
9g3f3Owf+P1u/5Lpq/V8/s3qRedH60P+RyV76B+rXbt/Zfyc88/xn5z+8f2D9oP7b7aGgfqd/1PQ
3+Q/Z77z/c/8L/zv7p86/5n/Z+I/wc/kvuY+Qj8h/ln+H/sf7tf4fjJdT/2HoEeq3zb/U/3f/Mfs
16Sn8x/dfUL6l/6f3AP5X/Rv9x/fvyF+dv+14gX1//efsz8Af84/v//t/yHu0f2n/t/0XoV/P/9B
/5P9F8Bf8z/tv/O+5vwjelMMr7gyEBK/kF9wZCAlfyC+4MhASv5BfcGQgJX8gvuDIQEr+QX3BkIC
V/IL7gyEBK/kF9wZCAlfyC7Ayjw8rJPWArp58txVRF1g02mvpDUwvgqL4L7gx/u056LojKLNRNiO
0XNtoN266sCxAMYNv/d/s5XzESTGR7RXS8nfn78jtKLHnWZJSv5BfcGQgJX7335R14l92V1L/zru
BbM/iBy84PxuDwTXP/c5gwiNQkZfbHi4mykoy4MhASv5BfcEkGCIi9mAWLtPrhm4OzvO0JhUJIFC
21YcchQ5b8cYLxGsulNDkHlugAvN9erjyZpKKVxNAQs4XjPwkuk9uEngGhT4kp7T9zn/nCy+xCdz
+K1ML4Ki+C+4MhARTm7lYG7uofKqU3jb6MRPXY+ue6k4G2cmf5/RwEVfNl/kMsCpJX2jRLRl1duM
jyQ3xYYSv5BfcGQgJX7t8ucNFFnuYsd5aPMA4Dw4u4VcztMPxtXxJ+9aFvNmRuut/S6MOJPgqL4L
7gyEBKsvCkyoEeNEhYTsmMNCx/4Q2uuju0/IT/TyuZxpFpqYXwVF8F9wZAHottIqeCuzlC/ESrN9
4nGOFvoODIQEr+QX3BkAei1h7d73H8EgKEbR7WOmzlc2F2ESsH7BBTC+CovgvuDIPFEu+SF3Ro+y
tchW9acyDkIFBe2ab+IOM3MQw9BZog2ECCxDy8Zo4jd5Ip2w44QEr+QX3BkH/m1ccuDw2aWDmIUe
X69fgI6Crx6Q+9QszXigMRpPxOAWfgqL4L7gyEBK+IK2ZB8ypMC2Iu+t1qafhZmw4bfdDXTnzrvd
k+RyC+4MhASrMnjVjqdNfI+GebOx2Yd3NDe7Xb1l4at32WoXn+80qDFDhY5SDe1J8ZcGQgJX8gvu
CQ/0BjzJVLQSHIwt8QR3V4xG8iFJmYmMbFfDuKUqfxTc4QEr+QX3BkH/lWR9ndLeC0edwDwDZu6V
gNLtugyK0CE+sISqbJ4KjBH6bhbwStdSEBK/kF9wZCAIJ+w0ivEdpQLVUee4Y65FVBzko1hJkT/1
HevqnCFDTqjMaIBhkRq7dxx0aT9isYR6UlnRxauNc+OxJHUCHHCAlfyC+4MhAB5go/KdgdGY6bHs
fU7gXYbj4f+1Ay0aPFgR/zzRnzRyoZCAlfyC8Ro8Jm5lztJ3pKQIG95ArGqGe/RRRey2uOfUcvrx
m2uW0iOFxyBD9wZCAlfyC9726xfzB8gQ4rW62jUa4vCt0zKKP1hf4y5d/sMjYi+C+4MhASv48vLu
s7RQxXeCPdZY7frMErz+OhKix9/V2phfBUXwX3BilMn0lcbpkF4kY6creIl5qnolj5rUp16AzP4i
MaHqgoDp5VHaVDIQEr+QXfKXRC+MpJHrSzwy12KWjRri8WclvyR54/q/+P0x0KykzDw5yLG11pkq
SlfyC+4MhARor4OOsEhQo3vDZX50Cb4p9fysKcF58mgAsFp45fBfcGQgJX8gu+UuvAEu4ZfSgfTo
me3+O8iuYkvzALjmMEASvQn1PjLgyEBK/kF9wSH+opAPSvvzATQ8bnBAX5v3E9aw/uTeBzlT2BHk
Qm0qGQgJX8gvt2EnY2zHHJpS8ZbDEN3TarjMx58+eQ9a8KaM/3WKp47CcOY/WZkpqiU/u+TQL8mU
a538aC6eJfO6GdSv5BfcGQgJX73EorfsYi6ag7ISODJEffp1bS6rf8799aQaNLVJ8JFtthvN59rM
H+f2WEgb4StdSEBK/kF9wZCAjRYR6tC2ZKdVb68muYkvP7zJ0L7LDVXcSPuvmya8Yae2h5JsP5a8
SuatI+ncJbkIG7h7hXUtPXMU4v8FRfBfcGQgJXylY0NYqp7IaJei5V1m6pfMtlnHemy6hZhtYsQU
wvgqL4L7gyDsqVpjVUN0ZRNGGsfqiK0s/TkCR1EJo792NCRgqi6tFfxUGQgJX8gvuDIO7zjhc3p4
HUpCYT8tOhicZTdTC+CovgvuDIP/XemnZNJhYcD/6cbRcU6ISshYEhBcvn6rX3EHwVF8F9wZCAlf
vffk08JK/lGkCVmT9U5gA1EretLcWrOeQ0jbA/1mRsqGQgJX8gvuDEu9gTh0V5sJpVfr72gkxGA/
1exwXymwWW0kQHB02E45mgKxOdWfTEMJyET/gu38HWEmKFaoxVLSfVX7jmzDbUUezLeiIiHW4n30
1A/QcOiu/yC+4MhASv5BfT62phBlpULaFCc4JpMLDgpeeHPFQbI5BfcGQgJX8gv/2dSj/pmZZcMD
xfBUXwX3BkICV/IL7gyEBK/kF9wZCAlfyC+4MhASv5BfcGQgJX8guoAA/v+pKAAAAAAAAAAAAHq1
EqY8EEsiDK4eNwK0U9sdNy519LtAZc4hzoeGGpgsAHs0+qndSp7bA+lQpQm+HnyAAy/mkDmL7kfT
2z4BTWrwgDysY9Y/2dhk0YcX5gh3wmO/mT4w2/lC8adIUFQ6nrWmJW5/GqZGkpR7SCOvaScJerXh
5BKDvsmIHOtDk0TOPZAG5mwsd0xv8WZsa5YBG8y29b8rCF4vK2J+9+IlOO3jLF/Auoq1xSdVdZQZ
iBNndub3OM7orTwlncsJSl/TuIM89vMPwX6FKbMFbtIZ1DBaTtrCgvbyaf0Yd0KXipkbUYkKGuKy
XtPFGf7FYqmNCafMtOaa8hurZX15WwqnJM2N4m0yqBB/reDViGS3GzC2sxp8/RMx+eg2XENcpRJX
eBsnOT5aMZNK1sCUTOlAEil8M1AMXdDSPqkeyoC8IEXexIRKTfDx0n6hzy2P8+WVEy3B876+V1Wt
jJ5QuSrN3GPej5qiw9tC6QpLZhBlQcx31DpdnuzwKQWfwZZKUz08QZVUehnv1YzKn7ICRCfauq52
+/83s49BTChp4I+DZolFvFYdZgc97xb8I1LNLwkierLIyJVGY8OKGdzvPnQFDDJMQxkSrNoAV4dx
aegIGhZk+fTj7HSZpE7tb/YmIPSn6Bj/yTG322fry/1ou4Qr7j0BHggRq9miNt4Oo8Gb6/QTs65x
887A2FAULWzwDvUfXgO//rMBYP+1t6oVtFl/kWhu9zGi+QkB+ylC5C7i8rH69Vo6GS5M4+OhR7xe
waUaxHXat+n81Yff+zoNbm81NMzPMNPY4PWmSMAu8OodKCu0MBhzmiQx7lVMRb1MYzBGma7vvkz+
MMZqJXZ+i51nCypFlFOXcE74FPledTE11AF+83LtIbFxvwtUg3Hsu6qBdrY5oXRncj/yy8sNTsey
4LBn9W5aBQnXbI8WybfrVU5s6F3PBr8qCLrl1nTXW63CNcZWo4caH/4JV4UaM9lbf01iJuOm303v
ayHOD4fzhReakiBcmfme8V1ZBNG1FqqrRV6m1w+TwChsQwPoFFgp1mt3kNAVIYUgnK4K3dmbZNKa
/lLZNSEQGAGpI5hwBNuh13t0RBDZ9YkQN+OQ0AZuL+0o7TqinQIlaCmZCfAC4pvn+MYHDE5lib7D
DuYKPTJavQPKzQ/uFgqQDRovXbfHhsGQR4XHU5spF4115hZ+T2zBJVq+9ctWh72IlMKLPG1Sxnf9
hcKMUyKKSfcl0iEtTv+PulEn6r6JJzrTEyjEM69ZtfXR7luY+Lw88LFecc/DNMZS68f4dNwmrcx8
uWCSBmkQke/pBL7l1ICtPq3jA3giOKPZdsLc7M9pe713EWQhp4VAHVXY5e0l7O5G/0xlFXm4J90c
lDo33eZnt5sG8cSR1OT5Uny+2dBx0hslgoPFk9Fn/fg7L6Si1ApKdsPbIj5aI7oGy4ONKOFqFRkV
ol/91zZSGWbtrWDYvtWHJtBlyvTTo2huzT2sfW0Dl3u5fnD05l/0MyzdLsHbt49S3ZYPJEZPHuKe
EoG60kxAYBbRT6eskZQljr79n6t7BubqEi6jXRUEiiHo7y6j9b60fKIYypsWfuUe5Cv78vzraR2S
2MTcuvDx429L8VLwz0l3ryOV/jO9P2aO2vimqmZm7sC9iaq4nskC1WRL+CN4Ty80v7+7CLeIYA+F
Bif/eBBfjEQrEGUDXoVvDiwXLTmXRQHeZNMiUHtPg/tTQAeS1c82ymfWbxfKPlj4FTuef+7rLN2z
ErLQErtIwDkXx5u9HxluG/+Cri6VDmUiSS2/FlpRM4GnDlRVbWewh9LzaPRde8j3ZVJGOG/BksGl
Elc2SlOqbOWckaMKCqQwmUwwgG9DEc0ZDD7jcxiCXvCylb3xIb9I7VfggEpQxBXeFfdtbjE73mIq
esPIt0SMUpKO2SakGIGm2kxC/QW2l10EY+Q9xULQMeYxuOX+kO4MlTUQSXWw22tETa+8km0s+IXf
QojVkY8iNjV3PUflK82jXz/odn7prADfMEwb52o34bGUEUpLIVUFZUDCPy6/LXXXk4avt/5O3Z8E
ZSTIeGU26idtT3mNTBwYGUXlFxIjr7iqadrzEIczZAM5l/FF9rMk5xv/rVeSRdwMajhzh2RMr5sm
illWKO9LbUs8+FFRZ+be8w+xFJ7WTJ0oh3p+s4cXd56jia9nHDH0bZn3xZPHPhgK3wBwMnVngwLs
oh8uCZ6DgC6fQFqHeFBR2vVOlY0dQg4500Z9KKv1WbZWO2KkmAACOsZ4EEZjTb7T4t+dR4bQiHlB
M2YSdDQXCrzP641pD6yEjJ0PRoKHsE956tGYOyXyeCJucr7K8YYGWPZbPfc/kuH5AM9/CpMtlQbx
Y+Xm0Nt6h+QgtcUgZOpXpEYVKMC4kSDgak7JmA5Mb6PiYcaC5fNeb8EiN+ByE9nBBRwheZbCkmlP
l/sTA5sPiZOEllz99+Vo3U55WNI2ud7lEjQy71vAEJFPD9lGjFsNxlrHRDFLhG/IxycmawEwW0f1
2/WA5KV4P1uGPFL5/Wd6JFhuYwJIp9/mikMZiLzMtaOYa91JaWw9Tt/zRf4UzpmzV3LmRtWrLbBI
GNTVUJgKDXfca/IsIPKVHa9itLgBjg4MTWYyHk/MJZCZMMu6gjsMtFl6CvOmnTz41QMIalm3bzW8
fgyyWWBSHJY5Lu5s9ZjYAHrlNDkt8jA+f68iX9nhCFQjEg1vzM+2MEIC545tIVWz5u0gEow7vi3o
BoqPmana1FwtsYL6215IbkodXPLDxs9M+W8g5Y3yM/wYI8N2KyH0fDrXX2XoDsic4scAb656+AO3
fjW84nOcMdNsjpehky+oJrIuQA8kLmBz0xWIoWb6OZWihBLlqiVosNej/g4HYvdvdpCWd1SJGyJG
W6E4PWBh9H+h3Gik6raL8OT/kZlz2v2v27z6SBugS1E0fdon3dUNkA/gEzrvkK/7MJc3xYT20xrh
U5S+pi/y2IB56mAM0mI4PS9KAGM/fPvapKfxiKTn0CiLaEA57feOqVnOserkb/57gXrc89bJ8Pwu
IhgBDWgBlhzyu+HYa5PsbQ8kvzvYPIHSQ1HUu12EiEx5KD8L3wI3CEv/CUkFYT8TRX8Vl+vGdD/C
gy4b8iR4acBvgegyjVBFminzoueaGbNLdK+EYs9URgSC2K3Z55CgUtYvgsr1pDMJ/HFvglKVDgw5
MxPrPc8wF2dY8cr35u87frj2VBO1gec5TQ/lVzgLPRf1ly3q2xBQJRsNHI/2VWvcuKbgWoa76QUs
NYDLouajXLw7oX7nYKHriKDLuI+hmUSyhXKK6OITqQWW1z2NzSLUVKhMlZO3JfQFkbBPzIWM/uK/
MTMesqd2AAAiolidmv9dhYdbmMkfJcChiF9btg6IKr4mv+KJyf7Z+EMsZicfz3k0e2MttkB3DZb4
bRaPeV4NkwvyAAXUept7iEgyeJokBnaO5YGUgLB4HRvmXKrx8HrUFAFrI1+4Lk2k/9s6vCcYLtqw
FCvPRhHRxWsHZNVpaHoOZ1EVYINja7Jcmb+lDkmRjGO0OeYb1viAWq8SCzBqxD3sxYrAW/Mg2fOv
KGrXRUNUe0pIvBVPaHWS8QGpNnX7dnlQCT7sEN/Q8gFP8ssthImswlcz7QAF9I2xCW1bRCABi3Ou
uVOHRy4LAi5PjbovUsm7nBrO6yu4A+pjgOkJdaVBD/0/7r4JfB65+4BOgQ1HNd8EbZVBC99rNave
7eTuXlrE863Q9vUSo6N4lLDpfX5nLY3Q8lCGYaL18fXIbCYrkx3JTHJc+QGoAQbVX/LkvGFvat4w
PAUido0E4T+PB9M2qJy9UAKa447Bp68jpHuw4LVrrCa+ziLzdvXIKkkyzWb0lyc8FaYUb9Dnta6W
PEXmzrvfZ7oOEDbWl/KaVeObubmA8BGXRRQTzJgoAOjT/S2SS3bVlxkRiUkXZi79t514rnHoEb62
nseOq7kc6oM2BH45zPBKBYbEZ4T9rWR0K3WlOtoBoNyFceSven8JrE+X/JEeNc9fcYabBnkkBRfe
/NAIgFVrytmZVXzCEx66pZ6FF3bIgdlgd5wSqF72wa+0U1YPvn+jKYjusOioScNjD3+PHF7Kf+pH
W3pD22fNDvuF56xm94fouyw2GbEElgw9FhabKG0qAr4km7Lfu3hs5e52jweVhgR7zdi58ZCsjTlz
2ls+jEQdKRk/oT1RPODILIvQYPy0Qxn0dQMIhbRvhGoLYXFPbP/+AgqCvn2B1kCO52TM+k8tstf+
LRjwVu6ORQYOVDZL+C9Q/EW5mLKh1HzvJhAAV9+xwvYZ+pApcGfslFlZ0rJ8nq806is+77ljQQ+i
gTTjwLCcTBu3A78dPVzaWly/GAbFO/8Py9idUQ0BfgWeDgeKIPOQSZgKsOLw+eYX/n/+cG8mYN/x
t2Dl/7mShbGi/2PIxnXrKmIRCfrM+TyFFO5l59r3Hiesr1WdiJGRY/xAYdZgjRv7pj7wv9dX4p7h
0HH8D7/EEADnjqi84Rf3ZAT5D2c2KHBYADdJvH22DpWONe68QWjUIWlHpxhd6DocpGkIYEeB5i0A
CwuH3sE/qYslGOl9DhlwPkYZez7KHYDe5JoBx4P4gqcONkSCP7JZKSQmfbJNdMPdlf9kU3Tw+Cf5
etqOrdhpYZ1BnseJpQMNH3KHF9Fkp/4bb05y4uMOqm0NocBDl/Pvv2ADrudZ/3O0bIAaLRsH2v7V
mgv0z9LErtn07+0vlKe5AzoieXSuWdkVG1UPA5S1sMs0wvJZywcNecBhikdVSUGTmiudmLf/xnQY
a/08OXCBCK4/UgEGgQKeOPqiWF4TE+LfP8KBYeH7n53hNNWFSJYJmn6NTkIdlsvUKbusYkEZclal
7f7nRCYzcX/xtBmNH5AYG7HGilh/vl1v51qwnS2G4Q/GRPtchl4f7fGWgKxP4tnn1uwwYk6g1Bbt
Yvetef116t8kE+CU/TniEAAUn9bDBctD+dLs025sLcXrwo3rodBP1gyO5DHSV5b5vWf+tkxGfzSs
IUyK1onszGG6RrZCABGLG+wGrbDwBi6sDvmQA1YQNI1X0zYP82h2K0Dds8yD/eHK+iaEDq9GEFII
lkafhA7oIpinIs1/VVgYn6Pb2HQBu+0Ob4M2mhxW83J3taEk0X/x5olMfg9ax+AoEVWUGhS4C/kd
bo6Ik3dRYaIyV6hLAp/oTBABxevyEFLkk+fAjUBd41W5tbbs1LxgcMEnY/zanPY2xy1y5WCDdIGx
VbyM9hyWQsjVv6ZEMVYBK5Q7wPXG4AAI5wLKHo81y8lH67We9d1ZnVA9+rlc7gI06v1sPm57daoQ
wiR92FqeR+ooe1Fpxo7mk5et87fKRdPUHqgF5l/Rn0fazzRRI+Vw9HyNXVSdIsClTIQC9hlwQnK5
RYkL2wVMIXe+Ng70raqHism0aLuovqnuBQCa2wdiJOOfi1jnM8DSGeUKO7RWhpeaxW7WIjXy32vW
LcQs8vJH71AduvC/clUrvtAl8/KZF5MGb6Yr8JGinx+IiAZ5BiUUHsyxm07aqmyXQHmSZUGSmntE
rV5CT+/myeX265toBIOs7t9azotGfeBZWJqa7CDZ7fsqtEY/P//yoYwnk5sESvkC14t4hTmacpta
h2Fp5qdTeMamnXzjQGBAAAslNInaETLyJdmAzex7UBUKzHFiin1vANGgjWCafNrF/krDYRKwUNgn
x3p8Pn7ebivh0qSZ9m/qUqMUBwosBIF4NtVECejnjP9qV9B0JtEKH73Za8L8hzASe1ev+6cWehoQ
AeYdpcg9Z+apnceIGak/HMD2VddS3xqH16/+8jCZKyCdYDk/h07XNl2kXB0gizIgKWFZPGh07h3W
F0pYtIiC8JlBUfiPdh2503QYjlmCnro6zjor8a4z+fQQ5aPpRk9+7qkbFmyuTNqY/Vo7MuzgtsaS
IIuWIvNYYdmw4QqfbY8eyrIgFuyOKTi+1cTrzUMH+xR2DrbF63+6ZgvvdoPyfCAbMtJDKQxLKTK5
6JA2WUvu4zUVbp89Kfx5k6B2JYgpQsDnPJdw4JJdmDeEbHM76zImCgx4/v1NcdywYEaXcIHnf+SO
qyaAo8Q1Gz2LQlbVfKn0ibaPNu0IkYr9vGgBC08JJBfG8Oh2hpkhCWiq+Yj/8eUaKyUS4/4cFf0u
1KLKR61zNPlIC+h4fAOhptbMkQhnRrsAhWjxATstCXOCzDc883/IXXbMGyXNF+Lulla4fW0Zb6b4
2K5SozAjm0EBLGnzqJcUIKYmNZw4BWcKfM7bejp7bu3E6yzYgI+hz/lF+YpQgBb9W76zLsSiZjxW
of9QmOS0oX+YqinXcKKnQYooeHc+BdjwYW5mw7HPXrv0R/Pwo+i1s58BwpeyVJEOGyTGyoC+StfX
V4YVp6t0xhYBFbnucQ0wFRRuN3LVA2gGL/FZxwgX/IH28gcGvHLUlWDySynEMWDDsUnP3VnZ8yli
KTP4y3cBdRLVS7CIp9w2JOnpGzsM27Lr7+D3z8sE34ACqiBEMju5wT2m7dH1Mq6yrxUp7Ds97iAa
M1//vsjwJ4GTTcBPPhYz8KYg2Eo2miSbLkhfgYVSAejmYYlgUyIeMvOSmOkp3ARTmlNvSPnbPEzq
+2mJXcqnJ6VM5ULdAP90qLkdzYHTRD/w1sqCKYFgzNGx0AeIL/tOhRkGAlA0tfdPndpe6gIbm7W+
70EbFTjccMt4ngPnvCgCEmrJ06KmgqeSY1poIfYKmphmfQ73poARG5NFvrAImk/sMsijVPGFRajm
+A/0zQHkIqExS27Za/QK6n+P7JMaPg87Unyus2IhbYvoc3Ku+E8Kq1nNrQAqr6gh9x/oad7sBpOF
ipkfUMH65/Y70nyfrTRHv1SNPqZYW9kueAy5sqLEP4a9hwii/TnFZfwC4QN7OpCQq7S2Abf5eGaQ
vIzBGCTGTHQ6rHw4JjWmt9j56ieE6jnnEdbKPpMi3nf37SiekVhAyq9L+fL78gcQkY1QBGrBNE6+
wi0dhrr4BNfwsGOaUf/8pLLWo3aDpr4ZdbbuGDTLzCAY1lf/UigdCrYKJMrFEiiN3h32jm7NjoWA
ZfV/16hABbxEv9Aox7C3D3LDsD27g1a50edDMgEQqDGJgI0uuVotnO9lmM009u1MgIbDvkquQ79m
hq6fHsy7b4RGFtVUa3gCR2mtLzugrYMAkgJuelvfUFjvIDGlRGOWvjlr4p5icPERYkGTIVQAOdEq
jxuGKAAA5JWTgQgK7tzhn7v/Ft8Heewf333I4lCvna3QFfFVKgkIi3DsUb5jMr3Kxcz5P/qI5l3r
DZvT/+Z1+C18JdAz5NEQeKcvwVGFJ9ZoWb84cIQSdWoL0kSbcwCDIQee1lfsxtULzQ9+BGCsp6Io
oM04iIHqMrgPKIu1rsxY3F1yHpYg0jius+NWvlC7dTcgZowvCWmdt2AAA8q1GHA7EpyF6buYpce8
AF1YOl87XKckARk/MwDa/Tt70/0NDn3ziyJwJPlCHMwbYJ1hPa4HTIkgCgSIJIDsSePvPh0I4+WR
L/B47x3cU+0V3seydXW4O1HFE+eQniYb1uQBl/ymsSsyHEpxJ+TecUKBlf21tMUADvqXfd+szMiJ
lsWT1ym8RW9hRyzAIyf5DNrAgDzaJIhw4KSW5GVbqPIdu2+I+4AuhyE4LRdGaaXPaFsXF8LS3C6y
9yF+SJg1+zoEEQHdT3ZvVXD/JR/mOpb/4jDRII/f/C6lLB7k8Su1dQX7zjti3gqGm99P1U44ACyd
DON30LIkfgrNK9B17dPr2r70e+Jkgor1yplKfxD8CaQyoJ+0OFjhZDd7szGKu/GUozi4JbgPFlL7
81SUifgOWaYU0bEUG1QUMeaLUvj3qoMDjsfop5m5/rjDV9Vgt75gDghl14EuC9aKBKGdGIAY1By+
ACGidpv+edg43LhqG3L+Kaag8OzNTitPN/z11ZcgbrAx+NdGds2VmA0PaDjw00D0FT8EXRG6kDzh
2oQkenYg6BTfOQKLcBdQ7C/N3SL3C4K+O1W/J/s4ILKtnpZI0J1rRHhNr5uZy5ETli65RGDej9Bl
NKOPe6CIpiZyv2ozbqNcY/4Hkxs8k3pJDlxK/PGni6reGQm3Z5D+K7yzG6ldWbbW0EKELQSxzkO1
pN9kXsXqjL06RFVRQEwsB0/kZXRbkHFzmSD4OSYBKxHwxGp+j1EW4NXBl8LXA45FSBlmMqTxHceg
OIW+SWsYXPgWEK0bo9y7K/tCeASkyNO2vA1SR9EsDPhR0Qi6Taln4/5LD5ACTDdmvVpysdmIh0Sm
Gjj59Rr6k0trVv8ar415YoVNb1tm569FO3ArkYeNBS5vZx78qotw0mPLBHRtPh96o3RBOpPh+roi
n+edkuizv+XZR6G1cM5EvpWLVtoOObiM74f2bD6brHQVJ3Lzhlz5Xsd/SD9HJu3bvp2ri/e0baXH
FBfGVkMe9A77m+QuN1FICq+AwEJePi4aJJDA8tTD8Us/ngAsEuiUXrZFDo4crZzJuuQJXkE09BMs
b81LzV7PlKdd3g9Bj8oPKU7M0Mc0UJkR8jwYcVdfJLEQbZhoRE03KY3cox7pRcjoCd1QOcSbxqQK
lp+lGTS8mJajhIE+lo5m/M5Fi73g/YhHXHz+S/qgdmkk+lK7Dx9bUb17Y79mjuXwWmo/zC1WaSrW
/Bc8HSytiHZ2zhN7dC3G/zsqnm2KKvz6PU/z+4KQ0lkPDgZkjU5h/m7J6voaOZvKSdB9J8nFKs33
3tvuTioG9wAFQoP/OxeMoG2MnVfVEMWnJmu0EgeGyCpQp4mMRTjgwZ++HOcU4HVu94+Vc890JCZN
LvNxNqgsY4BFB5IPYS1xkP8mkRxIP2b7RgiPVdVpdtP3VpBrA2fzCn1bE4dziawiDYStUAM6nByQ
szskN7lCN1KP9ga+30d0IBcvCDCykoTL5U4ZM/9Y7Z/hcRWfGmRSgc5+bKU0CP7P/mlp/dAmzb4k
okeNr/BPxIoh6i9Z6cOXqOwAIE0HYwouP5Zfv+za8G9gS5EkfsEfBs7A1ueO6WHYS2Ea7zIKJu/4
QDTfx/gl1BkkcJVn9FYP8i+D7P4cTN+t/Wd79sSYgkYg9z8eVKUHA6N/weZ7dS6JwCgi9jX/FziJ
MKNIkE75Mr5KpGLpG/fzf4U1N0Mvb7PuW/2erxNTQmNItd+/6DdP76dd5xO9aOdG3wb5+8lbmTaV
+W3QqaUuToTmwz94wrcnXv2Meifg2OGR5kiVY/cVuz96FgMIymqX5yal5ux65V+BjdbkkvmHOoIq
d8bNa1jstNI2xc+9pM4qXdMJC2AO2fNVzVcaygFiiQ3oeO+wrC+0wnyWik7KZW0hSXS+6gosCU7C
9ldrw2jt1evReqOnBBtYWGXASpDJPQc51VPncD1a1sBfYAdp7Ewog/h4WCxE6aAHrTQJnELshaGW
AzTa83fqKNVaQuYbkzqwdLh6m77533MCFrWY9+5PGxdzbVY/v8vF7l/FRYcPpJgT/+5h+pH52+Um
5TcOZEZ0Esi6SsQKchnIQl9zvR2XJYh3FSZgoRZ+cvGqfTM4m5YebArX/X/yWGsmD1XE09LIcgcJ
MiP/0wwToLcyxgr+caO3gC5x/WZB9RE+alAyoxnRrtbFZAvHvsq3tsOCrQ/G4K+DKQ0SIR5MYrde
8O32a/2CJ62gALqeaI8uySvgV4EaC9PpHakMDdcRHegSlpUQxK9vqk9TuBGj9U6nGhCBxZuYqf/p
MAnbEd1OBN305ZsdPeFLCqE4S7DfF1jlaTmFdgdTPa5C1mhGYdngleuoqqwUi9XdEHpNj5fsb/hM
bPgdxjglMVGpmp+bD93o8hf4f0ErSG1HOl+sFTEhVPcsg72lmZMKx/vDqFYv+N9JG5fEbMAMyF6G
hXdpqiGeU2qkX8e+2BFwygWtHUNpbB6KssPoeNESLrAk3NRZov7p/zb3iZRdLHhUjngn4mhmT81A
2jlltf7fkdCaM9tTcJr/cPugweDcOhR4XYMFdyxL4WdF3ULthEyDgGBrLEBRxhOSfnCJqSyrpmyR
SaJOYbsfz3GsgdWYeXScDE2v6pXiAC4XvBtfhFcKvxdNN0rn6Fiw/jCarYrqvS7lSXwcuZ8SMp/2
F8EGXBw7Xb6WM9UId4Yhn4DhD6euJRO2cuL58SwLkmReZiwozxxlgpBKi1QBcawa4xPCKcQ4kYAf
Wgd0Z8MkVdjF7PeivZhUiEwmXxKQWOLABKbEdXAAfWsKT2l23WRYL5FzbPk3CXt9yRJUA+wBwnJI
5V0ykyRaPzxCiaC3KogDZfv7j2huPhubNUASesGF2p9+w/kjxgdcMW/Ai72D20GU7xzywfjy+uQ1
Kkek4dJCzxxVjbRwJn1ERpeob2zj0H8ke5LgMGAEaf4yo/HhuOeqF/aRNXfmqTKlmtF0KOCX372x
NO/g19Q7jWDl4oaBMyDbvMvE9+1nF2Tzh8D4V5731hYay+F0V3PKHD86gGtY1o7jhMfX/99MM+v9
/FQquS+Hng/7x/4gVVJ6Nv7Y55VATEvQHluPsdY/O1SMijAbgm5/euJzutCvm+3dCJPnfWbRGue6
ubu8syhcOBdfpXAT2Y1poplmSM/TET5V9xEmaQRU2O0age29kuaK24xsiGID/g3WXJlNuq0u3uIq
1KH69lU4KtN9tIXyCs0IhdourLpUunYiU9FwaCE6hRmpSftfBTctIuzZtFuqXu0muPJ1SgLth8NM
fUleXehsjD1qD6gTQkjA7xhSNWD7+uD9WWMNUlRdB9cZ/6iu/kmsWBE/NIVSH90T5wibdu3+Kz3g
IVXOIYSXy0M0zvgxSU7UbJt7TEqbkw6kvzzE7j+XyRQIXSjULAzJ3c0PlBTqkA3XzMj6P0wOGQYU
iU0bPjqMH+TuIElQADaPlB4+JP1VRpuihKUisQWdlFvho7Y3kg6F0JJvUESU/52KsR+C+3BOQG7L
Lu6Iuf+qEtpxjhxArsnhIWz5/eGSdad7MQuHdlR9oItT9muVCMb2acvNehb+AHPVsVQJ4T7Y91Rc
KiMnxcBGdduGf+z2v+brM9FTSEvrmPDf5szDZ0XUsnrnwiW2zm3Ct2eBe7az6+IWNZY2lVIFvTue
6vcKeC0VbacGQ5n961FvM+L0/Xnnt4ukUOOneXSF92XUrkWPcWH/T5FWYKzvGadhGtr6IE+ZcEjf
Z/MgTwdvwaALUyQZjk/eOBkGWuQ5laX0u8ulwSI6fR754OPz8/N8bLO9wCECzvoe4USDS8Oh57NX
LL29RDwA7bT0t0scdfs8eIjvRYdNKZJtyH3wSo46Uo9ERdLzMLf761MLG7TVlZdWIs+VNZIT5tPc
SpUGKaELu0DLud0wdYEOWpMGOQ8H8+UWsFc2QTUUt87BTqsh+YveZSLk7+3dgKv40+w3sRzAmVbO
DKO4k1hQmJZQZWDjOrAPrZeyuQ51u+tD+ITUOUSLgELTQByC6uccFu7m0eDApU6BxNHNyzV3hgz3
vgAAAAAnRt0Xqi7Cf7zHC8RwZICuCHhNhzOmUHb/ZtLVw9f5UxcJQn9MIL9c+EX+Q0GI/77IVNTB
XKKceG4KFtgNPwMXsyk440BcckedacbFCrDDgUFT/RNHM1XiIUqOWNFLDG3rd4BgJLDkEWID2/V9
LyGQ78m+Bh4W5BwfngdJa2QAAsfOLxJJdehd3r6k1f/WK+TCfrvncKFq+qxua694ktoT1CgQJjfH
DizRmSMr0cB4MVqGxLxzPV3WddhIlFM7BcSeegBBeRaSv85Kcb9dMsuCPwtxcFFHRRqr4l4i3xUd
wzeQw2PetwIgqj6nYowWmB5Ems3BMEf8L/SaGR8h0zk3v8/WGyBnuX1KVAC9hL7/4K+xgQcB5REq
13Pm02/MY9rbj7qA25Cc6CMAdtu/4wdqgFiO/dcKez+ZQ/6abTJWVT/7/9vDMOBEZUY/0FxJCXyf
vghcDx9LLx4mOqaIaK+5yqcAOcyliyv9vDekR6PAX7dhtF49hpQIu3NKcNclE/N6BvYRfuuEifw5
V46iVhZd6TzMH2ti3piwf3L5U6cSOVx6bE8rvP0g8k2s9Z2gEIU+n4SfiIFKDIppC38qEiKz3zHS
kaqrAO2dHfxzNG+LyrTQQOmz3fq8auAq7JFQwNtnDrEtFeYzEYYOUCPH7/UxKlNzvcsDraPaa2WV
QHn+IycjUJCzOuh+5oYspP9RBfzrJDS5nBEf6KcIBOO/3IIo1Mh644QPkIwGI76A4A05AbvC7RKi
uPUVHcAtTlaNT6Q895q1RNOwslYtfR0cIjscC1wY6Q7LIyMRxZtsmVgseWpragOKSwz+R3+c76tg
qWk04vBN4CdLuAjNXo6JKkrmU4F8pviq2dLuAga1FTL0oU7veqLS8Ha6FGJcaaAJexTOpWYG8F6P
A0Pr/Ztn+wEFWcu5BTQfvNC11PA1s6Z0lCK/qPjuqk2hO6oG5r+QASj5dGpGDibiDyJWbYJY/O6M
0SxRSofj+KJ9Y3lEZDfnt/G15ZNuSZfPEPIb52IX6+TvIeXBJ5DnJSK1JUvmKwIX07omoxTytYuB
kzQxlT6d7nbl08ahSVqAhgziOeKzFw68M4Rdgb8OCLAVDsMNWhpB+qkh7M1BnfwjN/nfgQ+oeUyU
RqUY5IS7u2hy0/0P9PY3413lxtBBSUWXV5VyGnrcgCzu01oG9iiY9tevLNXbnld750G8lAf5CDeq
T9ENkeKw1BXHK0unhAtpNbyMTOAxc1W2SmwStllAYBRecKSmrjG9JmMXiuo+cApYhlTS9zOv7SQR
ce3FbqeQv/FeehgG4ff7qtqNTKxfkcH80o9bKbNwxjtSEOU6Z02PeYUztCtDyiYeD3vVQlhR4lLG
k+OfS1eHn4oK3GpeZWouqGsPOB43bydkp/4Y1Fi4wFjfSYZbWV8cs2od10OafYSonOqgpnkU2VXh
YabhFJdrKDnyduirG/HNgVBniGWSxWKYVOZAqg7n4RqMbrpa5ZtToC8VAWQ+IIKBmy8cWm8XlyAa
zapPtbS52ezY6oQmi6rAfkSiAAIv7iArdObRA+DhRddmXWz0WCkc7hAKCh0I6t0lDZ15d/kKLnpW
n0jUNN3rVAR3PwsbNgq3stQSGkV59sjbFA2tJfiVodhXyocSMoJZlVY2auAEAkglVhRrEtKGo/oJ
SfyoLhn0Ab899sQuq0wLM+1OTaxC/oaCvCeQjceXbxer1x0LshKa4ip8c/5E2gbKgKrTjQ2YbRfQ
vVCCiS/3exQJUBsmbxHE2MajX4I9Sx1UV00/9miYggA3I6H+FGfJ3pcJ7Ys3JWj8Ge2Idnl/2owW
+TBZltW3tS9NBsMwgd2szUe4a++18ukKFCHBv6bezHUSXBfOskJGYGijx2EAl5J5g96es2PKSkYd
0zXPPsdmwxBTlgX0B4ccGXIaRQliASLxWgCfMv5q2t1U9S4AAAAAAAAAAAAAAAAAAAAAAEVYSUa6
AAAARXhpZgAASUkqAAgAAAAGABIBAwABAAAAAQAAABoBBQABAAAAVgAAABsBBQABAAAAXgAAACgB
AwABAAAAAgAAABMCAwABAAAAAQAAAGmHBAABAAAAZgAAAAAAAABIAAAAAQAAAEgAAAABAAAABgAA
kAcABAAAADAyMTABkQcABAAAAAECAwAAoAcABAAAADAxMDABoAMAAQAAAP//AAACoAQAAQAAACUC
AAADoAQAAQAAAIgCAAAAAAAA



In [ ]:
self.embedding =3D torch.nn.Embedding(vocab_size, attention_dim)self.positional_embedding =3D torch.nn.Embedding(context_length, attenti=
on_dim)


In [ ]:
embedd=
ings =3D self.embedding(context)context_len =3D context.shape[1]position =3D torch.arange(context_len, devi=
ce=3Dcontext.device).unsqueeze(0)pos=
ition_embeddings =3D self.positional_embedding(position)e =3D embed=
dings + position_embeddings


In [ ]:
self.w_key =3D torch.nn.Linear(embed_dim, attention_dim, bias=3Dbia=
s)self.w_query =3D torch.nn.Linear(emb=
ed_dim, attention_dim, bias=3Dbias)self.w_val=
ue =3D torch.nn.Linear(embed_dim, attention_dim, bias=3Dbias)=


In [ ]:
k =3D self.w_key(x)=
   # (B, T, A)q =3D self.w_query(x)=
 # (B, T, A)v =3D self.w_value(x) # (B, T, A)```where,B: batch size,T: context length,A: attention =
dimension```


In [ ]:
scores =3D (q @ k.transpose(-2, -1)) / (k.size(-1) ** 0.5)  # (B, T, T)


In [ ]:
mask =3D torch.triu(torch.ones(T, T, device=3Dx=
.device), diagonal=3D1).bool()scores =3D scores.masked_fill(mask, float('-1e10'))attn =3D scores.softmax(dim=3D-1)  # (B, T, T)final =3D attn @ =
v  # (B, T, A)


In [ ]:
class SelfAttention(torch.nn.Module):    def __in=
it__(self, embed_dim, attention_dim, bia=
s=3DFalse, dropout=3D0.1):        supe=
r().__init__()        self.w_key =3D torch.nn.Linear(embed_dim, =
attention_dim, bias=3Dbias)        self.w_query =3D torch.nn.Linear(emb=
ed_dim, attention_dim, bias=3Dbias)        self.w_value =3D torch.nn.Li=
near(embed_dim, attention_dim, bias=3Dbias)        self.dropout =3D tor=
ch.nn.Dropout(dropout)    def forward(self, x):        B, T, _ =3D x.size()        k =3D se=
lf.w_key(x)   # (B, T, A)        q =
=3D self.w_query(x) # (B, T, A)    =
    v =3D self.w_value(x) # (B, T, A)        # Scaled dot-product attention        scores =3D (q @ k.transpose(-2=
, -1)) / (k.size(-1) ** 0.5)  # (B, T, T)        # Causal mask (future positions masked)        mask =3D=
 torch.triu(torch.ones(T, T, device=3Dx.device), diagonal=3D1).bool()     =
   scores =3D scores.masked_fill(mask, float('-1e10'))        attn =3D=
 scores.softmax(dim=3D-1)  # (B, T, T)        attn =3D self.dropout(a=
ttn)        return attn @ v  # (B, T, A)


In [ ]:
class MultiHeadAttenti=
on(torch.nn.Module):    def =
__init__(self, num_heads, embed_dim, attention_dim, dropout=3D0.1):        sup=
er().__init__()        self.head_size =3D attention_dim//num_hea=
ds        self.heads =3D torch.nn.ModuleList()        for i in range(num_heads):            self.heads=
.append(SelfAttention(embed_dim=3Dembed_dim, attention_dim=3Dself.head_size=
,dropout=3Ddropout))    def forward(self,x):        head_outputs =3D []        for head in self.he=
ads:            head_outputs.append(head(x)) #B x T x A//num_heads        concatenated =3D torch.cat(head_=
outputs, dim =3D 2)        return concatenated


In [ ]:
class Decoder(torch.nn.Module):    def __init__(self,num_heads,embed_dim,attention_dim, d=
ropout=3D0.1):        super().__init__()        self.masked_mul=
tihead =3D MultiHeadAttention(num_heads, embed_dim, attention_dim, dropout)=
        self.feed_forward =3D FeedForward(attention_dim)        sel=
f.n1 =3D torch.nn.LayerNorm(attention_dim)        self.n2 =3D torch.nn.=
LayerNorm(attention_dim)    def forward(self,x):        e =3D self.masked_multihead(self.n1(x))    =
    e =3D  e + x        e =3D self.feed_forward(self.n2(e))        =
return e


In [ ]:
def top_k_logits(logits, k)=
:    v, ix =3D torch.topk(logits, =
k)    out =3D logits.clone()  =
  out[out &lt; v[:, [-1]]] =3D float('-inf'=
)    return outdef generate(model, max_new_tokens, context, context_length, temperature=3D1.0, top_k=3DNone):    res =3D []    for _ =
in range(max_new_tokens):        i=
f context.shape[1] &gt; context_length:            context =3D context[:, -context_length:]        logi=
ts =3D model(context)  # [B, T, V]=
        logits =3D logits[:, -1, :]  # [=
B, V]        logits =3D logits / max(temperature, 1e-3)        if top_k is not None:            logits=
 =3D top_k_logits(logits, top_k)        if torch.isnan(logits).any() or torch.isinf(logits).any():     =
       raise ValueError("Logits contain NaN or Inf")        probabilit=
ies =3D nn.functional.softmax(logits, =
dim=3D-1)        probabilities =3D t=
orch.clamp(probabilities, min=3D1e-9, max=
=3D1.0)        next_token =3D to=
rch.multinomial(probabilities, 1)  # [B, 1]=
        context =3D torch.cat((con=
text, next_token), dim=3D1) =
   return contextstart_context =3D "I want something"model =3D GP=
T(num_heads,vocab_size,embed_dim,attention_dim,num_blocks,context_le=
ngth, dropout_rate).to(device)mode=
l.eval()token_ids =3D generate(    model=3Dmodel,    context=3D=
text_to_token_ids(start_context, token=
izer, device),    max_new_tokens=3D10,    context_length=3Dcontext_length)print("Output text:\n", token_ids_to_text(token_ids, tokenizer))


In [ ]:
Output text: I=
 want something introduce=E3=82=A6 coaches Kard Judais=
m trendsCommerce rotating i=
nfiltration approach=
